Todos los imports

In [ ]:
!pip install folium --quiet

import os
import random
import re
import time
import zipfile
from abc import ABC, abstractmethod
from datetime import datetime, timedelta
from urllib.parse import quote

import folium
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests

Celda para llamar a la API de datos abiertos de Tenerife, se ha dejado ahí para ilustrar cómo se haría

In [ ]:
#import requests
#import zipfile
#import o

#API_URL = "https://datos.tenerife.es/ckan/api/action/package_show?id=36c2e26f-0d18-4b5a-b214-1636168e0765"

#leer la API
#response = requests.get(API_URL)
#response.raise_for_status()
#data = response.json()

#comprobar que la API respondió bien
#if not data.get("success"):
#  raise ValueError("La API no respondió bien")

#zip_url = data["result"]["resources"][0]["url"]
#zip_path = "titsa_gtfs.zip"

#descargar el zip
#response = requests.get(zip_url)
#response.raise_for_status()

#with open(zip_path, "wb") as f:
  #f.write(response.content)

  #print("Zip descargado en:", zip_path)

Descargar los datos del gtfs desde el github, cargar los datos de los POI y función fix_coords, esta función se desarrolló para corregir el formato de las coordenadas, ya que al extraerlas de las apis de google places y foursquare salían en formatos diferentes

In [ ]:
ZIP_URL = "https://github.com/lauglezg/DatosTfg/releases/download/gtfs/fichero-zip-de-google-transit.3.zip"
zip_path = "titsa_gtfs.zip"

ZIP_URL2 = "https://github.com/lauglezg/DatosTfg/raw/main/fichero-zip-de-google-transit-tranvia.zip"
zip_path2 = "tranvia_gtfs.zip"

response = requests.get(ZIP_URL)
response.raise_for_status()

with open(zip_path, "wb") as f:
    f.write(response.content)

print("Zip descargado en:", zip_path)
response = requests.get(ZIP_URL2)
response.raise_for_status()

with open(zip_path2, "wb") as f:
    f.write(response.content)

print("Zip descargado en:", zip_path2)

#----------------------------------------------------------------
URL_REPO = "https://raw.githubusercontent.com/lauglezg/DatosTfg/main/"
#pois_info = "/content/drive/MyDrive/TFG-Laura González González/Pois_25 - poi_info.csv"
#pois_time ="/content/drive/MyDrive/TFG-Laura González González/Pois_25 - time_open.csv"

#poi_info_df = pd.read_csv(pois_info, dtype={"poi_lat": str, "poi_lon": str}, decimal=",")
#poi_time_df = pd.read_csv(pois_time, decimal=",")
#poi_info_df = pd.read_csv(URL_REPO + quote("Pois_22 - S_C_poi_info.csv"), dtype={"poi_lat": str, "poi_lon": str}, decimal=",")
#poi_time_df = pd.read_csv(URL_REPO + quote("Pois_22 - S_C_time_open.csv"), decimal=",")

poi_info_df = pd.read_csv(URL_REPO + quote("Pois_38 - S_C_LL_poi_info.csv"),   # hace falta normalizar la coma
                          dtype={"poi_lat": str, "poi_lon": str},
                          decimal=",")
poi_time_df = pd.read_csv(URL_REPO + quote("Pois_38 - S_C_LL_time_open.csv"),
                          decimal=",")

#--------------------------------------------------------------------------------------
def fix_coord(val): # Función para corregir el formato de las coordenadas
    s = str(val).strip()
    if s in ('nan', 'None', ''):
        return np.nan
    negative = s.startswith('-')
    digits = re.sub(r'[^\d]', '', s)
    if not digits or len(digits) < 3:
        return np.nan
    fixed = digits[:2] + '.' + digits[2:]
    return float(('-' if negative else '') + fixed)

# Aplicar la corrección
poi_info_df["poi_lat"] = poi_info_df["poi_lat"].apply(fix_coord)
poi_info_df["poi_lon"] = poi_info_df["poi_lon"].apply(fix_coord)

Zip descargado en: titsa_gtfs.zip
Zip descargado en: tranvia_gtfs.zip


Se combinan los dos csv para tener mejor la información

In [ ]:
pois_all_info = poi_info_df.merge(
    poi_time_df.drop(columns=["poi_name"]),
    on="poi_id")

def detect_municipality(direccion):
    """Deduce el municipio a partir de la dirección del POI."""
    d = str(direccion).lower()
    if "laguna" in d:
        return "La Laguna"
    if "santa cruz" in d:
        return "Santa Cruz"
    return "Otro"

pois_all_info["municipio"] = pois_all_info["poi_address"].apply(detect_municipality)

Esta función obtiene el horario del poi, para poder incluirlo o no en la matriz de destino, en función del día de la semana

In [ ]:
def  get_open_hours(open_hours, day):
  weekend_days = {"saturday", "sunday"}
  result = []

  for poi_id, group in open_hours.groupby("poi_id"): #Caso 1: cuando cierran uno o varios días a la semana, que no sea fin de semana
    single_day = group[group["poi_day"] == day]

    if not single_day.empty:
      if (single_day["status"] == "close").any():
        continue

      result.append(single_day[single_day["status"] == "open"])
      continue

    if day in weekend_days: #Caso 2: cierra los fines de semana
      weekend = group[group["poi_day"] == "weekend"]

      if not weekend.empty:
        if (weekend["status"] == "close").any():
          continue

        result.append(weekend[weekend["status"] == "open"])
        continue

    default = group[group["poi_day"] == "default"] #Caso 3: Resto de dias

    if not default.empty:
      if (default["status"] == "close").any():
        continue

      result.append(default[default["status"] == "open"])

  if not result:
    return open_hours.iloc[0:0].copy()

  return pd.concat(result, ignore_index=True)

Se extrae la información de los GTFS

In [ ]:
gtfs_zip_path = "/content/titsa_gtfs.zip"
gtfs_folder = "/content/titsa_gtfs"

os.makedirs(gtfs_folder, exist_ok=True)

with zipfile.ZipFile(gtfs_zip_path, "r") as zip_ref:
  zip_ref.extractall(gtfs_folder)

  print("GTFS descomprimido en:", gtfs_folder)
  print(os.listdir(gtfs_folder))

tranvia_zip_path = "/content/tranvia_gtfs.zip"
tranvia_folder = "/content/tranvia_gtfs"

os.makedirs(tranvia_folder, exist_ok=True)

with zipfile.ZipFile(tranvia_zip_path, "r") as zip_ref:
    zip_ref.extractall(tranvia_folder)

    print("GTFS del tranvía descomprimido en:", tranvia_folder)
    print(os.listdir(tranvia_folder))

GTFS descomprimido en: /content/titsa_gtfs
['stop_times.txt', 'trips.txt', 'stops.txt', 'routes.txt', 'calendar_dates.txt', 'agency.txt', 'shapes.txt']
GTFS del tranvía descomprimido en: /content/tranvia_gtfs
['stop_times.txt', 'trips.txt', 'calendar.txt', 'transfers.txt', 'stops.txt', 'routes.txt', 'calendar_dates.txt', 'agency.txt', 'shapes.txt', 'frequencies.txt']


Juntando las guaguas y el tranvía en un solo mapa de datos

In [ ]:
# Cargamos los archivos del GTFS de TITSA
stops = pd.read_csv(os.path.join(gtfs_folder, "stops.txt"))
routes = pd.read_csv(os.path.join(gtfs_folder, "routes.txt"))
trips = pd.read_csv(os.path.join(gtfs_folder, "trips.txt"))
stop_times = pd.read_csv(os.path.join(gtfs_folder, "stop_times.txt"))
calendar = pd.read_csv(os.path.join(gtfs_folder, "calendar_dates.txt"))

# Cargamos los archivos del GTFS TRANVÍA (Metrotenerife)

tram_stops      = pd.read_csv(os.path.join(tranvia_folder, "stops.txt"))
tram_routes     = pd.read_csv(os.path.join(tranvia_folder, "routes.txt"))
tram_trips      = pd.read_csv(os.path.join(tranvia_folder, "trips.txt"))
tram_stop_times = pd.read_csv(os.path.join(tranvia_folder, "stop_times.txt"))
tram_freq       = pd.read_csv(os.path.join(tranvia_folder, "frequencies.txt"))
tram_calendar   = pd.read_csv(os.path.join(tranvia_folder, "calendar.txt"))

def hhmmss_to_min(s):
    h, m, seg = str(s).split(":")
    return int(h) * 60 + int(m) + int(seg) / 60.0

def min_to_hhmmss(x):
    x = int(round(x))
    return f"{x//60:02d}:{x%60:02d}:00"

# Calcular el horario del tranvía
expanded_rows = []
for _, trip in tram_trips.iterrows():
    template_id = trip["trip_id"]
    template_stop_times = tram_stop_times[
        tram_stop_times["trip_id"] == template_id
    ].sort_values("stop_sequence")
    if template_stop_times.empty:
        continue

    # Tiempo de cada parada respecto a la salida del viaje.
    base_min = hhmmss_to_min(template_stop_times.iloc[0]["departure_time"])
    stop_offsets = [
        (row["stop_id"], row["stop_sequence"],
         hhmmss_to_min(row["arrival_time"]) - base_min,
         hhmmss_to_min(row["departure_time"]) - base_min)
        for _, row in template_stop_times.iterrows()
    ]

    trip_counter = 0
    for _, freq in tram_freq[tram_freq["trip_id"] == template_id].iterrows():
        window_start = hhmmss_to_min(freq["start_time"])
        window_end = hhmmss_to_min(freq["end_time"])
        headway_min = freq["headway_secs"] / 60.0
        departure = window_start
        while departure < window_end:
            new_trip_id = f"{template_id}_{trip_counter}"
            for stop_id, stop_sequence, arr_offset, dep_offset in stop_offsets:
                expanded_rows.append({
                    "trip_id": new_trip_id,
                    "arrival_time": min_to_hhmmss(departure + arr_offset),
                    "departure_time": min_to_hhmmss(departure + dep_offset),
                    "stop_id": stop_id,
                    "stop_sequence": stop_sequence,
                    "template_id": template_id,
                })
            trip_counter += 1
            departure += headway_min

tram_st_exp = pd.DataFrame(expanded_rows)
trip_to_route_service = tram_trips.set_index("trip_id")[
    ["route_id", "service_id"]
].to_dict("index")
tram_tr_exp = (
    tram_st_exp[["trip_id", "template_id"]].drop_duplicates()
    .assign(
        route_id=lambda d: d["template_id"].map(lambda x: trip_to_route_service[x]["route_id"]),
        service_id=lambda d: d["template_id"].map(lambda x: trip_to_route_service[x]["service_id"]),
    )
    [["trip_id", "route_id", "service_id"]]
)

# Prefijo TRAM_ para que los IDs no choquen con titsa
TRAM_PREFIX = "TRAM_"
tram_stops["stop_id"]       = TRAM_PREFIX + tram_stops["stop_id"].astype(str)
tram_routes["route_id"]     = TRAM_PREFIX + tram_routes["route_id"].astype(str)
tram_st_exp["stop_id"]      = TRAM_PREFIX + tram_st_exp["stop_id"].astype(str)
tram_st_exp["trip_id"]      = TRAM_PREFIX + tram_st_exp["trip_id"].astype(str)
tram_tr_exp["trip_id"]      = TRAM_PREFIX + tram_tr_exp["trip_id"].astype(str)
tram_tr_exp["route_id"]     = TRAM_PREFIX + tram_tr_exp["route_id"].astype(str)
tram_tr_exp["service_id"]   = TRAM_PREFIX + tram_tr_exp["service_id"].astype(str)
tram_calendar["service_id"] = TRAM_PREFIX + tram_calendar["service_id"].astype(str)

# Calendario del tranvía: días de semana -> fechas (formato calendar_dates de titsa)
def tram_calendar_dates(DATE):
    d = datetime.strptime(str(DATE), "%Y%m%d")
    dias = ["monday","tuesday","wednesday","thursday","friday","saturday","sunday"]
    col = dias[d.weekday()]
    activos = tram_calendar[tram_calendar[col] == 1]["service_id"].tolist()
    return pd.DataFrame({"service_id": activos, "date": int(DATE), "exception_type": 1})

#  Combinar con el GTFS de titsa
stops = pd.concat([stops, tram_stops[["stop_id","stop_name","stop_lat","stop_lon"]]],
                  ignore_index=True)
routes = pd.concat([routes, tram_routes[["route_id","route_short_name",
                                          "route_long_name","route_type"]]],
                   ignore_index=True)
trips = pd.concat([trips, tram_tr_exp], ignore_index=True)
stop_times = pd.concat([stop_times, tram_st_exp.drop(columns=["template_id"])],
                       ignore_index=True)

print("GTFS combinado (TITSA + tranvía):")
print(f"  paradas:    {len(stops)}")
print(f"  rutas:      {len(routes)}  (tranvía route_type=0: {(routes['route_type']==0).sum()})")
print(f"  viajes:     {trips['trip_id'].nunique()}")
print(f"  stop_times: {len(stop_times)}")

GTFS combinado (TITSA + tranvía):
  paradas:    3817
  rutas:      178  (tranvía route_type=0: 2)
  viajes:     97600
  stop_times: 2726855


Relacionar todos los ficheros para que diga la linea de la guagua, el horario, la parada

In [ ]:
transport_info = (
    stop_times
    .merge(trips [["trip_id", "route_id", "service_id"]], on="trip_id")
    .merge(routes[["route_id", "route_short_name", "route_long_name"]], on="route_id")
    .merge(stops[["stop_id", "stop_name", "stop_lat", "stop_lon"]], on="stop_id")
)

# Comprobar que el tranvía está en transport_info
tram = transport_info[transport_info["route_id"].astype(str).str.startswith("TRAM_")]
print("Filas de tranvía en transport_info:", len(tram))
print(tram[["stop_name", "route_short_name", "arrival_time", "departure_time"]].head(5))

Filas de tranvía en transport_info: 17238
              stop_name route_short_name arrival_time departure_time
2709617  Intercambiador               L1     06:00:00       06:00:00
2709618       Fundación               L1     06:02:00       06:02:00
2709619  Teatro Guimerá               L1     06:04:00       06:04:00
2709620          Weyler               L1     06:06:00       06:06:00
2709621          La Paz               L1     06:07:00       06:07:00


Calcular distancias

In [ ]:
def l1_distance(lat1, lon1, lat2, lon2):
  a = np.abs(lat2-lat1)
  b = np.abs(lon2-lon1)
  return (a+b)*111

# Matriz de distancia entre pois
def build_distance_matrix(coordenadas):
  n = len(coordenadas)
  matrix = np.zeros((n, n))
  for i in range(n):
      for j in range(n):
          matrix[i][j] = l1_distance(coordenadas[i][0], coordenadas[i][1],
                                      coordenadas[j][0], coordenadas[j][1])
  return matrix

Convertir a números la latitud y longitud para que no tener errores

In [ ]:
poi_info_df["poi_lat"] = pd.to_numeric(poi_info_df["poi_lat"], errors="coerce")
poi_info_df["poi_lon"] = pd.to_numeric(poi_info_df["poi_lon"], errors="coerce")
pois_all_info["poi_lat"] = pd.to_numeric(pois_all_info["poi_lat"], errors="coerce")
pois_all_info["poi_lon"] = pd.to_numeric(pois_all_info["poi_lon"], errors="coerce")

Buscar las paradas más cercanas a cada poi

In [ ]:
def nearby_stops(poi_lat, poi_lon, stops_df, max_walk_min=15, velocidad_pie=4/60):
    """Devuelve paradas  a <= max_walk_min caminando del POI."""
    max_dist_km = max_walk_min * velocidad_pie
    dists = l1_distance(poi_lat, poi_lon,
                        stops_df["stop_lat"].values,
                        stops_df["stop_lon"].values)
    mask = dists <= max_dist_km
    res = stops_df[mask].copy()
    res["dist_km"]  = dists[mask]
    res["walk_min"] = res["dist_km"] / velocidad_pie
    # quedarse con una fila por parada (la más cercana) y ordenar
    res = (res.sort_values("walk_min")
              .drop_duplicates(subset="stop_id", keep="first")
              .reset_index(drop=True))
    return res

Pasar de horas a minutos, construir ventanas de tiempo y comprobar si los pois están abiertos

In [ ]:
def to_minutes(h):   # Pasar de horas a minutos
    if pd.isna(h):
        return None

    h = str(h).strip()
    hour, minute = h.split(":")
    return int(hour) * 60 + int(minute)

def build_time_windows(open_pois_df):
    """Para cada poi_id, lista de franjas (apertura, cierre) en minutos."""
    ventanas = {}
    for _, fila in open_pois_df.iterrows():
        if fila["status"] != "open":
            continue
        pid   = fila["poi_id"]
        abre  = to_minutes(fila["poi_open_time"])
        cierra = to_minutes(fila["poi_close_time"])
        if abre is None or cierra is None:
            continue
        ventanas.setdefault(pid, []).append((abre, cierra))
    return ventanas

def is_open(idx_poi, hora_llegada, ventanas_horario):  # Para comprobar si la ruta está correcta con respecto a los horarios de entrada
    """True si el POI idx_poi está abierto a hora_llegada (en minutos del día)."""
    franjas = ventanas_horario.get(idx_poi)
    if not franjas:
        return True   # sin horario conocido, no se restringe
    return any(abre <= hora_llegada <= cierra for abre, cierra in franjas)


Celda del GRASP

$$
\begin{array}{l}
\textbf{GRASP Construction Algorithm: }  \\
\hline
\textbf{Input: } POIs, route, size \\
\textbf{Output: } \text{route} \\
\textbf{function } \text{ConstructGRASP}(route, POIs, size ) \\
\quad cur\_route \gets route \\
\quad CL \gets POIs \\
\quad \textbf{while } CL \ne \emptyset \textbf{ do} \\
\quad \quad (a) \text{ Eliminar de } CL \text{ los candidatos sin inserción factible en } route \\
\quad \quad \quad \textbf{ para cada } poi \textbf{ in } CL: \\
\quad \quad \quad \quad \text{ Encontrar la mejor insercion de } poi \textbf{ en } route \\
\quad \quad \quad \quad \textbf{ If } \text{(no hay insercion factible de } poi \textbf{ en } route) \\
\quad \quad \quad \quad \quad \text{ eliminar } poi \textbf{ de } CL \\
\quad \quad (b) \text{ Construir } RCL \textbf{ con los } size \textbf{ mejores } pois \text{ de } CL\\
\quad \quad \quad \textbf{ para cada } poi \textbf{ in } CL: \\
\quad \quad \quad \quad \textbf{ If } \text{(RCL no tiene tamaño } size) \\
\quad \quad \quad \quad \quad \text{ añadir } poi \textbf{ a } RCL \\
\quad \quad \quad \quad \textbf{ Else }  \\
\quad \quad \quad \quad \textbf{ If } \text{(El peor poi de RCL es peor que } poi) \\
\quad \quad \quad \quad \quad \quad \text{ reemplazar } poi \textbf{ en } RCL \\
\quad \quad (c) \text{ Elegir un } poi \textbf{ al azar de } RCL \\
\quad \quad (d) \text{ Insertar } poi \textbf{ en } route \\
\quad \textbf{endwhile } \\
\quad \textbf{return } cur\_route \\
\textbf{end function} \\
\hline
\end{array}
$$


In [ ]:
# Calculcar la puntuación de una única ruta
def compute_route_score(route, scores):
    """
    Calcula la puntuación total de una ruta sumando los scores de cada POI en la ruta.

    Parámetros:
        route (list[int]): Índices de los POIs en la ruta.
        scores (list[float]): puntuaciones de los POIS.

    Retorna:
        float: Puntuación total de la ruta.
    """
    return sum(scores[i] for i in route)

def evaluate_route(r, init_time, travel_time, visit_times,
                   initio=0, final=0, init_hour=540, franja_min=30):
    """Comprueba horarios y calcula la duración en un solo recorrido.

    Retorna:
        (bool, float): si la ruta respeta las ventanas horarias y su
        duración total; (False, inf) en cuanto un POI está cerrado.
    """
    n_franja = travel_time.shape[0]

    def to_franja(t):
        if not np.isfinite(t):
            return n_franja - 1
        return int(np.clip((t - init_hour) // franja_min, 0, n_franja - 1))

    t = init_time
    anterior = initio
    for nodo in r:
        t += travel_time[to_franja(t), anterior, nodo]
        if not is_open(nodo, t, ventanas_horario):
            return False, np.inf
        t += visit_times[nodo]
        anterior = nodo
    t += travel_time[to_franja(t), anterior, final]
    return True, t - init_time


# Calcular duración total de una ruta
def compute_route_time(route, init_time, travel_time, visit_times,
                       initio=0, final=0,
                       init_hour=540, franja_min=30):
    """
    Calcula el tiempo de llegada de la ruta incluyendo traslados y tiempo de visita.
    La ruta empieza en initio y acaba en final; que pueden o no ser el mismo
    La lista "route" sólo tiene POIs
    Parámetros:
        route (list[int]): ruta actual con índices de POIs.
        travel_time (list[list[int]]):  tiempos entre POIs.
        travel_time[t, i, j]: es el tiempo en que se llega al POI j
                             partiendo del POI i en el instante t
        visit_times (list[list[float]]): tiempo estimado de visita por POI.
        init_time (init): instante de inicio (en minutos)
        initio (int): punto de inicio de la ruta, por defecto el hotel
        final (int): punto final de la ruta, por defecto el hotel

    Retorna:
        float: tiempo de llegada de la ruta.
    """
    n_franja = travel_time.shape[0]  # Devuelve la hora (en minutos del día) a la que termina la ruta

    def to_franja(t): # convierte una hora del día al índice de slot correspondiente
        if not np.isfinite(t):
            return n_franja - 1   # tiempo infinito o NaN: devolvemos la última franja
        return int(np.clip((t - init_hour) // franja_min, 0, n_franja - 1))

    if len(route) == 0:  #Si la ruta está vacía va directamente de start a finish
        if initio == final:
            return 0
        else:
            return travel_time[to_franja(init_time), initio, final]

    trip_time = travel_time[to_franja(init_time), initio, route[0]] + visit_times[route[0]]
    # tiempo inicial más tiempo de primera visita

    for i in range(1, len(route)):  # tramos intermedios entre POIs
        trip_time += travel_time[to_franja(init_time + trip_time), route[i - 1], route[i]] + visit_times[route[i]]

    trip_time += travel_time[to_franja(init_time + trip_time), route[-1], final] # vuelta al hotel
    return trip_time


def arrival_time_at_position(route, pos, k, init_time, travel_time, visit_times,
                             initio=0, init_hour=540, franja_min=30):
    """Hora EXACTA de llegada al POI k si se inserta en la posición 'pos'."""
    n_franja = travel_time.shape[0]

    def to_franja(t):
        if not np.isfinite(t):
            return n_franja - 1
        return int(np.clip((t - init_hour) // franja_min, 0, n_franja - 1))

    nueva = route[:pos] + [k] + route[pos:]
    t = init_time
    anterior = initio
    for nodo in nueva:
        t += travel_time[to_franja(t), anterior, nodo]   # llegada a poi
        if nodo == k:
            return t                                       # hora de llegada a k
        t += visit_times[nodo]
        anterior = nodo
    return t

def simulate_insertion(k, route, pos, init_time, travel_time, visit_times,
                       initio=0, final=0, init_hour=540, franja_min=30):
    """Simula insertar el POI k en la posición pos de la ruta.

    Recorre la ruta resultante una sola vez y devuelve a la vez la
    duración total y la hora de llegada a k (antes hacían falta dos
    recorridos separados para obtener ambos valores).

    Retorna:
        (float, float): duración total de la ruta en minutos y hora
        de llegada al POI k (en minutos del día).
    """
    n_franja = travel_time.shape[0]

    def to_franja(t):
        if not np.isfinite(t):
            return n_franja - 1
        return int(np.clip((t - init_hour) // franja_min, 0, n_franja - 1))

    nueva = route[:pos] + [k] + route[pos:]
    t = init_time
    llegada_k = init_time
    anterior = initio
    for nodo in nueva:
        t += travel_time[to_franja(t), anterior, nodo]
        if nodo == k:
            llegada_k = t
        if not is_open(nodo, t, ventanas_horario):
            return np.inf, llegada_k, False    # Si el POi está cerrado
        t += visit_times[nodo]
        anterior = nodo
    t += travel_time[to_franja(t), anterior, final]
    return t - init_time, llegada_k, True


def find_best_insertion(k, route, init_time, travel_time, visit_times,
                        initio=0, final=0, init_hour=540, franja_min=30):
    """Busca la mejor posición para insertar el POI k en la ruta.

    Prueba primero la inserción al final y después el resto de
    posiciones; a igualdad de duración se conserva la primera evaluada.

    Parámetros:
        k (int): índice del POI candidato.
        route (list[int]): ruta actual.
        travel_time (np.ndarray): matriz 3D de tiempos (franja, i, j).
        visit_times (np.ndarray): tiempo de visita de cada POI.

    Retorna:
        (int, float, float): mejor posición, duración total de la ruta
        con k insertado y hora de llegada al POI k.
    """
    best_pos, best_time, best_llegada = None, np.inf, None
    for pos in range(len(route) + 1):
        new_time, llegada, factible = simulate_insertion(
            k, route, pos, init_time, travel_time, visit_times,
            initio, final, init_hour, franja_min)
        if factible and new_time < best_time:
            best_pos, best_time, best_llegada = pos, new_time, llegada
    return best_pos, best_time, best_llegada

#================================================================================================================
# GRASP para construir UNA ruta
#================================================================================================================

def construct_route_grasp(POIs, travel_time, init_time, max_time, max_per_category,
                          candidate_pois, init_hour=540, franja_min=30,
                          route=None, initio=0, final=0,
                          rcl_size=3, restricted_pois=None, required_pois=None):
    """
    Construye UNA ruta con GRASP constructivo usando eficiencia como función greedy del GRASP

# La ruta se construye desde rutas vacías o no; según el caso
# Pero si hay parte del recorrido ya hecho y se produce una interrupcción "initio" es el punto de se encuentra la ruta

    Parámetros:
        POIs: List of available POIs
        # contiene los todos datos de todos los POIs; aquí faltaría los TW (Early,Late)
        # Los POIs están numerados de 1 a n_pois; el índice 0 se reserva para el hotel
        trip_time (list[list[float]]): matriz de tiempos entre POIs.
        init_time int: tiempo de inicio
        max_time (float): Tiempo máximo total en minutos.
        max_per_category (dict(int)): Restricción de número máximo de visitas por categoría.

        route : ruta inicial que puede no ser vacía
        candidate_pois: pois candidatos para esta construcción
        initio,final : inicio y fin de la ruta
        rcl_size (int): Tamaño máximo de la lista de candidatos restringida
        restricted_pois (list[int] | None): POIs que no se pueden usar como candidatos; parámetro de la instancia
        required_pois (list[int] | None): POIs requeridos para esa ruta

        Primero se ejecuta GRASP sólo con los POIs requeridos, que da las rutas iniciales con las que empezar

    Retorna:
        list[int]: lista de POIs de la ruta o lista de índices
    """
    route = list(route) if route else []
    candidate_pois = list(candidate_pois)
    restricted_pois = restricted_pois or []
    required_pois = required_pois or []

    if restricted_pois:                       # quitar Pois restringidos de los candidatos
        for i in restricted_pois:
            if i in candidate_pois:
                candidate_pois.remove(i)

    if required_pois:
      for poi in required_pois:
          if poi in route:
              if poi in candidate_pois:
                  candidate_pois.remove(poi)
              continue

          best_pos, _, _ = find_best_insertion(
              poi, route, init_time, travel_time, visit_times,
              initio, final, init_hour, franja_min)
          if best_pos is None:
              print(f"Aviso: el POI obligatorio {poi} no cabe en la ruta con sus horarios.")
          else:
              route.insert(best_pos, poi)

          if poi in candidate_pois:
              candidate_pois.remove(poi)

    category_count = {}                       # contador de visitas por categoría
    for poi in route:
        cat = category[poi]
        category_count[cat] = category_count.get(cat, 0) + 1

    current_time = compute_route_time(route, init_time, travel_time, visit_times,
                                          initio, final, init_hour, franja_min)

    while candidate_pois:
        rcl = []
        rcl_positions = []
        rcl_times = []
        rcl_scores = []


        for candidate in candidate_pois:
            best_pos, best_time, best_llegada = find_best_insertion(
                candidate, route, init_time, travel_time, visit_times,
                initio, final, init_hour, franja_min)

            if best_pos is None or best_time > max_time:
                continue

            cat = category[candidate]
            if category_count.get(cat, 0) >= max_per_category.get(cat, float('inf')):
                continue

            delta = best_time - current_time
            if delta <= 0:
                efficiency = float('inf')
            else:
                efficiency = scores[candidate] / delta

            if len(rcl) < rcl_size:
                rcl.append(candidate)
                rcl_positions.append(best_pos)
                rcl_times.append(best_time)
                rcl_scores.append(efficiency)
            else:
                min_eff_index = rcl_scores.index(min(rcl_scores))
                if efficiency > rcl_scores[min_eff_index]:
                    rcl[min_eff_index] = candidate
                    rcl_positions[min_eff_index] = best_pos
                    rcl_times[min_eff_index] = best_time
                    rcl_scores[min_eff_index] = efficiency

        if not rcl:
            break

        selected = random.randint(0, len(rcl) - 1)
        poi = rcl[selected]
        cat = category[poi]
        insert_pos = rcl_positions[selected]

        route.insert(insert_pos, poi)
        candidate_pois.remove(poi)
        current_time = rcl_times[selected]

        category_count[cat] = category_count.get(cat, 0) + 1
        if category_count[cat] >= max_per_category.get(cat, float('inf')):
            candidate_pois = [p for p in candidate_pois if category[p] != cat]

    return route

Celda VNS

$$
\begin{array}{l}
\textbf{VNS-GRASP Algorithm: }  \\
\hline
\textbf{Input: } k_{max}, rep, size \\
\textbf{Output: } \text{route} \\
\textbf{function } \text{VNS-GRASP}(route, k_{max}, rep, size ) \\
\quad cur\_route \gets route \\
\quad k,r \gets 1,0 \\
\quad \textbf{while } k \le k_{max} \textbf{ do} \\
\quad \quad new\_route \gets Destroy(cur\_route,k) \\
\quad \quad new\_route \gets Construct(new\_route,\alpha) \\
\quad \quad \textbf{if } score(new\_route) > score(cur\_route)\\
\quad \quad \quad cur\_route \gets new\_route \\
\quad \quad \quad k,r \gets 1,0 \\
\quad \quad \textbf{else} \\
\quad \quad \quad \textbf{if } r < rep \\
\quad \quad \quad \quad r \gets r+1 \\
\quad \quad \quad \textbf{else} \\
\quad \quad \quad \quad \quad k \gets k+1 \\
\quad \quad \quad \quad \quad r \gets 0 \\
\quad \quad \quad \textbf{endif } \\
\quad \quad \textbf{endif } \\
\quad \textbf{endwhile } \\
\quad \textbf{return } cur\_route \\
\textbf{end function} \\
\hline
\end{array}
$$

In [ ]:
#================================================================================================================
# VNS extendido para mejorar UNA ruta
#================================================================================================================

def improve_vns(route, POIs, travel_time, init_time, max_time, max_per_category,
                candidate_pois, initio=0, final=0,
                k_max=5, repetitions=10, rcl_size=3,
                restricted_pois=None, required_pois=None,
                init_hour=540, franja_min=30):
    """
    Mejora UNA ruta con VNS constructivo-destructivo usando destroy aleatorio y repair del GRASP

    Parámetros:
        route: La ruta de entrada
        POIs: List of available POIs
        # contiene los todos datos de todos los POIs; aquí faltaría los TW (Early,Late)
        # Los POIs están numerados de 1 a n_pois; el índice 0 se reserva para el hotel
        trip_time (list[list[float]]): matriz de tiempos entre POIs.
        init_time int: tiempo de inicio
        max_time (float): Tiempo máximo total en minutos.
        max_per_category (dict(int)): Restricción de número máximo de visitas por categoría.

        candidate_pois: pois candidatos para la mejora
        initio,final : inicio y fin de la ruta
        K_max (int): Tamaño máximo de la destrcción
        restricted_pois (list[int] | None): POIs que no se pueden usar como candidatos; parámetro de la instancia
        required_pois (list[int] | None): POIs requeridos para esa ruta

        Primero se ejecuta GRASP sólo con los POIs requeridos, que da las rutas iniciales con las que empezar

    Retorna:
        list[int]: lista de POIs de la ruta o lista de índices
    """

    route = list(route) if route else []
    base_candidates = list(candidate_pois)
    restricted_pois = restricted_pois or []
    required_pois   = required_pois or []

    n_franja = travel_time.shape[0]

    def to_franja(t):
        if not np.isfinite(t):
            return n_franja - 1
        return int(np.clip((t - init_hour) // franja_min, 0, n_franja - 1))

    def route_time_of(r):
        return compute_route_time(
            r, init_time, travel_time, visit_times,
            initio, final, init_hour, franja_min
        )

    def respects_schedule(r):
        t = init_time
        anterior = initio
        for nodo in r:
            t += travel_time[to_franja(t), anterior, nodo]
            if not is_open(nodo, t, ventanas_horario):
                return False
            t += visit_times[nodo]
            anterior = nodo
        return True

    def local_search_order(r):
        mejor = list(r)
        mejor_t = route_time_of(mejor)
        hubo_mejora = True

        while hubo_mejora:
            hubo_mejora = False
            for a in range(len(mejor) - 1):
                for b in range(a + 1, len(mejor)):
                    candidata = mejor[:a] + mejor[a:b+1][::-1] + mejor[b+1:]

                    ok, t_cand = evaluate_route(candidata, init_time, travel_time,
                                                visit_times, initio, final,
                                                init_hour, franja_min)
                    if not ok:
                        continue

                    if t_cand < mejor_t - 1e-9:
                        mejor = candidata
                        mejor_t = t_cand
                        hubo_mejora = True

        return mejor, mejor_t

    global_score = compute_route_score(route, scores)
    global_time = route_time_of(route)

    k = 1
    rep = 0

    while k <= k_max:

        if len(route) <= k:
            break

        new_route = list(route)
        selected = random.sample(new_route, k)

        for j in selected:
            new_route.remove(j)

        cand_iter = [p for p in base_candidates if p not in new_route]

        improved_route = construct_route_grasp(
            POIs, travel_time, init_time, max_time, max_per_category,
            candidate_pois=cand_iter,
            init_hour=init_hour, franja_min=franja_min,
            route=new_route,
            initio=initio, final=final,
            rcl_size=rcl_size,
            restricted_pois=restricted_pois,
            required_pois=required_pois,
        )

        # La búsqueda local solo reordena (no cambia el score)
        new_score = compute_route_score(improved_route, scores)
        if new_score < global_score - 1e-9:
            if rep < repetitions:
                rep += 1
            else:
                k += 1
                rep = 0
            continue

        improved_route, improved_time = local_search_order(improved_route)

        mejora_score = new_score > global_score
        mismo_score_menos_tiempo = (
            abs(new_score - global_score) < 1e-9
            and improved_time < global_time
        )

        if mejora_score or mismo_score_menos_tiempo:
            route = list(improved_route)
            global_score = new_score
            global_time = improved_time
            k = 1
            rep = 0
        else:
            if rep < repetitions:
                rep += 1
            else:
                k += 1
                rep = 0

    return route

Celda VNS básico

In [ ]:
#================================================================================================================
# VNS para mejorar una ruta
#================================================================================================================

def improve_vns_basic(route, POIs, travel_time, init_time, max_time, max_per_category,
                 candidate_pois, initio=0, final=0,
                 k_max=5, rcl_size=3,
                 restricted_pois=None, required_pois=None,
                 init_hour=540, franja_min=30):
    """
      Mejora UNA ruta con VNS básico.
      Esta variante no realiza reintentos con el mismo valor de k:
      si no se obtiene mejora, pasa directamente al siguiente entorno.

        k = 1
        mientras k <= k_max:
            x'  = shake(x, k)         # perturbar: destroy k POIs + repair GRASP
            x'' = local_search(x')      # búsqueda local: reordenacion
            si x'' mejora a x (score, o mismo score y menos tiempo):
                x = x'' ; k = 1           # mover (intensificar)
            si no:
                k = k + 1                 # cambiar de entorno (diversificar)

    Devuelve la mejor ruta encontrada.
    """
    n_franja = travel_time.shape[0]

    def to_franja(t):
        if not np.isfinite(t):
            return n_franja - 1
        return int(np.clip((t - init_hour) // franja_min, 0, n_franja - 1))

    def respects_schedule(r):
        t = init_time
        anterior = initio
        for nodo in r:
            t += travel_time[to_franja(t), anterior, nodo]
            if not is_open(nodo, t, ventanas_horario):
                return False
            t += visit_times[nodo]
            anterior = nodo
        return True

    def route_time_of(r):
        return compute_route_time(r, init_time, travel_time, visit_times,
                                  initio, final, init_hour, franja_min)
    # Shake -> destruye k POIs al azar y repara la ruta con GRASP.
    def shake(r, k):
        nueva = list(r)
        fuera = random.sample(nueva, k)
        for j in fuera:
            nueva.remove(j)
        cand = [p for p in base_candidates if p not in nueva]
        return construct_route_grasp(
            POIs, travel_time, init_time, max_time, max_per_category,
            candidate_pois=cand,
            init_hour=init_hour, franja_min=franja_min,
            route=nueva, initio=initio, final=final,
            rcl_size=rcl_size,
            restricted_pois=restricted_pois,
            required_pois=required_pois,
        )

    # Busqueda local, mejora el ORDEN invirtiendo tramos, respetando las ventanas horarias.
    def local_search(r):
        mejor = list(r)
        mejor_t = route_time_of(mejor)
        hubo_mejora = True
        while hubo_mejora:
            hubo_mejora = False
            for a in range(len(mejor) - 1):
                for b in range(a + 1, len(mejor)):
                    candidata = mejor[:a] + mejor[a:b+1][::-1] + mejor[b+1:]
                    ok, t_cand = evaluate_route(candidata, init_time, travel_time,
                                                visit_times, initio, final,
                                                init_hour, franja_min)
                    if not ok:
                        continue
                    if t_cand < mejor_t - 1e-9:
                        mejor, mejor_t = candidata, t_cand
                        hubo_mejora = True
        return mejor, mejor_t

    # Inicialización
    route = list(route) if route else []
    base_candidates = list(candidate_pois)
    restricted_pois = restricted_pois or []
    required_pois   = required_pois or []

    global_score = compute_route_score(route, scores)
    global_time  = route_time_of(route)

    #  Bucle principal VNS
    k = 1
    while k <= k_max:
        if len(route) <= k:
            break
        x1 = shake(route, k)

        # La búsqueda local solo reordena (no cambia el score): si el shake dio peor score, la ruta será rechazada igual
        if compute_route_score(x1, scores) < global_score - 1e-9:
            k += 1
            continue

        x2, x2_time = local_search(x1)

        x2_score = compute_route_score(x2, scores) # Aquí se decide si se reordena o no
        mejora_score        = x2_score > global_score
        igual_score_y_menos = (abs(x2_score - global_score) < 1e-9
                               and x2_time < global_time)

        if mejora_score or igual_score_y_menos:
            route        = list(x2)
            global_score = x2_score
            global_time  = x2_time
            k = 1
        else:
            k += 1
    return route

Buscar la mejor conexión entre dos paradas

In [ ]:
# Guardamos los horarios agrupados por parada para no repetir el trabajo cada vez.
# Se rehace solo si cambia el día (cambia el nº de filas).
cache_paradas = {"n_filas": None, "salidas_por_parada": None, "llegadas_por_parada": None}

def prepare_indices(stop_times_dia):
    # Mismo nº de filas = mismo día = caché válida
    if cache_paradas["n_filas"] == len(stop_times_dia):
        return

    columnas_salida  = ["trip_id", "departure_min", "stop_sequence"]
    columnas_llegada = ["trip_id", "arrival_min", "stop_sequence"]

    # Agrupamos por parada una sola vez
    cache_paradas["salidas_por_parada"] = {
        parada: grupo[columnas_salida].values
        for parada, grupo in stop_times_dia.groupby("stop_id")
    }
    cache_paradas["llegadas_por_parada"] = {
        parada: grupo[columnas_llegada].values
        for parada, grupo in stop_times_dia.groupby("stop_id")
    }
    cache_paradas["n_filas"] = len(stop_times_dia)


def best_guagua_info(parada_origen, parada_destino, hora_salida, stop_times_dia):
    """
    Busca el viaje (guagua o tranvía) que lleva de la parada de origen a la de
    destino llegando lo antes posible. Devuelve un diccionario con el viaje o
    None si no hay ninguno válido a partir de la hora indicada.
    """
    prepare_indices(stop_times_dia)

    salidas_origen   = cache_paradas["salidas_por_parada"].get(parada_origen)
    llegadas_destino = cache_paradas["llegadas_por_parada"].get(parada_destino)
    if salidas_origen is None or llegadas_destino is None:
        return None

    # Solo las salidas a partir de nuestra hora (columna 1 = departure_min)
    salidas_origen = salidas_origen[salidas_origen[:, 1] >= hora_salida]
    if salidas_origen.shape[0] == 0:
        return None

    # Salida más temprana de cada viaje
    salida_de_cada_viaje = {}
    for viaje, minuto_salida, secuencia_origen in salidas_origen:
        if viaje not in salida_de_cada_viaje or minuto_salida < salida_de_cada_viaje[viaje][0]:
            salida_de_cada_viaje[viaje] = (minuto_salida, secuencia_origen)

    # Viajes que paran en el destino, en sentido correcto y sin tiempos negativos.
    # Nos quedamos con el que llega antes.
    mejor_viaje = None
    for viaje, minuto_llegada, secuencia_destino in llegadas_destino:
        if viaje in salida_de_cada_viaje:
            minuto_salida, secuencia_origen = salida_de_cada_viaje[viaje]
            sentido_correcto = secuencia_destino > secuencia_origen
            llegada_despues  = minuto_llegada >= minuto_salida
            if sentido_correcto and llegada_despues:
                if mejor_viaje is None or minuto_llegada < mejor_viaje["llegada"]:
                    mejor_viaje = {
                        "trip_id": viaje,
                        "seq_o":   int(secuencia_origen),
                        "seq_d":   int(secuencia_destino),
                        "salida":  int(minuto_salida),
                        "llegada": int(minuto_llegada),
                    }
    return mejor_viaje


# Pasamos las horas "HH:MM:SS" a minutos, que es mejor para comparar horarios
def time_transport_to_min(t):
    try:
        h, m, s = str(t).strip().split(":")
        return int(h) * 60 + int(m) + int(s) // 60
    except (ValueError, AttributeError):
        return np.nan

stop_times["arrival_min"]   = stop_times["arrival_time"].apply(time_transport_to_min)
stop_times["departure_min"] = stop_times["departure_time"].apply(time_transport_to_min)

# Añadimos route_id para saber a qué línea pertenece cada paso
stop_times = stop_times.merge(trips[["trip_id", "route_id"]], on="trip_id")


def time_guagua(stop_id_origen, stop_id_dest, departure_time, stop_times_df):
    # Versión sencilla que devuelve solo el tiempo total (espera + viaje)
    # Guaguas que salen del origen a partir de nuestra hora
    guaguas_origen = stop_times_df[
        (stop_times_df["stop_id"] == stop_id_origen) &
        (stop_times_df["departure_min"] >= departure_time)
    ][["trip_id", "departure_min", "stop_sequence"]].rename(
        columns={"departure_min": "salida", "stop_sequence": "seq_origen"}
    )
    if guaguas_origen.empty:
        return np.nan

    # Las que pasan por el destino
    guaguas_dest = stop_times_df[
        stop_times_df["stop_id"] == stop_id_dest
    ][["trip_id", "arrival_min", "stop_sequence"]].rename(
        columns={"arrival_min": "llegada", "stop_sequence": "seq_destino"}
    )

    # Cruzamos por trip_id y nos quedamos con los que van en el sentido bueno
    guaguas_origen_dest = guaguas_origen.merge(guaguas_dest, on="trip_id")
    guaguas_origen_dest = guaguas_origen_dest[
        guaguas_origen_dest["seq_destino"] > guaguas_origen_dest["seq_origen"]
    ]
    if guaguas_origen_dest.empty:
        return np.nan

    # La que llega antes
    best_guagua = guaguas_origen_dest.loc[guaguas_origen_dest["llegada"].idxmin()]
    waiting_time = best_guagua["salida"] - departure_time
    trip_time = best_guagua["llegada"] - best_guagua["salida"]
    return waiting_time + trip_time

Decidir desde dónde empieza la ruta según el hotel

In [ ]:
def prepare_hotel(name_hotel, municipio="Santa Cruz"):
    """
    Prepara el punto de partida de la ruta a partir del hotel elegido y del
    municipio que se va a visitar.

    La cercanía y la etapa interurbana se calculan respecto a los pois del municipio de
    destino, no respecto a todos. Así, un hotel de Santa Cruz que
    quiere visitar La Laguna se detecta como lejos y se busca el transporte
    interurbano que acerque a La Laguna.

    Deja listas las globales pois_con_origen y nearby_per_poi.
    Devuelve el tiempo de caminata inicial (walk_hotel_to_pc_min).
    """
    global pois_con_origen, nearby_per_poi, HOTEL_CERCA, walk_hotel_to_pc_min, parada_pc, parada_sc, parada_pc_id, parada_sc_id, HOTEL_LAT, HOTEL_LON

    pois_ = pois_all_info.drop_duplicates("poi_id").reset_index(drop=True)

    # POIs del municipio destino (para medir cercanía y centro respecto a ELLOS)
    if municipio == "Santa Cruz":
        municipios_ok = {"Santa Cruz"}
    elif municipio == "La Laguna":
        municipios_ok = {"La Laguna"}
    else:
        municipios_ok = {"Santa Cruz", "La Laguna"}
    pois_destino = pois_[pois_["municipio"].isin(municipios_ok)]
    if pois_destino.empty:
        pois_destino = pois_  # por si acaso, no debería pasar

    HOTEL_LAT, HOTEL_LON = HOTELES[name_hotel]
    print(f"Hotel seleccionado: {name_hotel}  ({HOTEL_LAT}, {HOTEL_LON})")

    VELOCIDAD_PIE = 4/60

    # Centro de los POIs DEL MUNICIPIO destino
    poi_centro_lat = pois_destino["poi_lat"].mean()
    poi_centro_lon = pois_destino["poi_lon"].mean()

    # Cercanía medida al POI más cercano DEL MUNICIPIO destino
    distancias_a_pois = l1_distance(HOTEL_LAT, HOTEL_LON,
                                    pois_destino["poi_lat"].values,
                                    pois_destino["poi_lon"].values)
    dist_poi_mas_cercano = distancias_a_pois.min()

    UMBRAL_CAMINAR_KM = 2.0
    HOTEL_CERCA = dist_poi_mas_cercano <= UMBRAL_CAMINAR_KM

    print(f"POI más cercano del municipio destino: {dist_poi_mas_cercano:.2f} km")
    print("Hotel CERCA: se empieza caminando (sin guagua interurbana)" if HOTEL_CERCA
          else "Hotel LEJOS: se necesita transporte interurbano para empezar")

    stops_ = stops.copy()
    stops_["dist_hotel_km"] = l1_distance(HOTEL_LAT, HOTEL_LON,
                                          stops_["stop_lat"].values, stops_["stop_lon"].values)
    stops_["dist_pois_km"]  = l1_distance(poi_centro_lat, poi_centro_lon,
                                          stops_["stop_lat"].values, stops_["stop_lon"].values)

    if HOTEL_CERCA:
        origen_lat, origen_lon = HOTEL_LAT, HOTEL_LON
        name_origen = f"Hotel ({name_hotel})"
        walk_hotel_to_pc_min = 0.0
        cercanas_hotel = nearby_stops(HOTEL_LAT, HOTEL_LON, stops, max_walk_min=15)
        if cercanas_hotel.empty:
            cercanas_hotel = nearby_stops(HOTEL_LAT, HOTEL_LON, stops, max_walk_min=30)
        parada_sc_id = cercanas_hotel.iloc[0]["stop_id"]
        parada_pc_id = parada_sc_id
        print(f"Origen = hotel. Parada más cercana: {parada_sc_id}")
    else:
        RADIO_HOTEL_KM = 2.0
        RADIO_POIS_KM  = 3.0
        ids_cerca_hotel = set(stops_[stops_["dist_hotel_km"] <= RADIO_HOTEL_KM]["stop_id"])
        ids_cerca_pois  = set(stops_[stops_["dist_pois_km"]  <= RADIO_POIS_KM ]["stop_id"])

        trips_h = stop_times[stop_times["stop_id"].isin(ids_cerca_hotel)][
            ["trip_id", "stop_id", "stop_sequence"]
        ].rename(columns={"stop_id": "stop_hotel", "stop_sequence": "seq_hotel"})
        trips_p = stop_times[stop_times["stop_id"].isin(ids_cerca_pois)][
            ["trip_id", "stop_id", "stop_sequence"]
        ].rename(columns={"stop_id": "stop_pois", "stop_sequence": "seq_pois"})

        candidatos = trips_h.merge(trips_p, on="trip_id")
        candidatos = candidatos[candidatos["seq_pois"] > candidatos["seq_hotel"]]
        if candidatos.empty:
            raise ValueError(
                f"No se encontró transporte directo del hotel a {municipio}. "
                "Prueba con más horas o con otro hotel.")

        elegida = candidatos.merge(stops_[["stop_id", "dist_pois_km"]],
                                   left_on="stop_pois", right_on="stop_id"
                                   ).sort_values("dist_pois_km").iloc[0]
        parada_pc_id = elegida["stop_hotel"]
        parada_sc_id = elegida["stop_pois"]
        trip_usado   = elegida["trip_id"]
        parada_pc = stops_[stops_["stop_id"] == parada_pc_id].iloc[0]
        parada_sc = stops_[stops_["stop_id"] == parada_sc_id].iloc[0]
        origen_lat = parada_sc["stop_lat"]
        origen_lon = parada_sc["stop_lon"]
        name_origen = f"Parada llegada ({parada_sc['stop_name']})"

        route_id_usado = trips[trips["trip_id"] == trip_usado]["route_id"].iloc[0]
        linea_usada    = routes[routes["route_id"] == route_id_usado]["route_short_name"].iloc[0]
        print(f"Transporte interurbano: línea {linea_usada}")
        print(f"  Salida : {parada_pc['stop_name']} ({parada_pc['dist_hotel_km']:.2f} km del hotel)")
        print(f"  Llegada: {parada_sc['stop_name']} (cerca de los POIs de {municipio})")
        walk_hotel_to_pc_min = parada_pc["dist_hotel_km"] / VELOCIDAD_PIE
        print(f"Caminata hotel - parada salida: {walk_hotel_to_pc_min:.1f} min")

    pois_con_origen = pd.concat([
        pd.DataFrame([{
            "poi_id": 0, "poi_name": name_origen,
            "poi_lat": origen_lat, "poi_lon": origen_lon,
            "poi_score": 0, "poi_cat": "none", "poi_visit_time": 0,
        }]),
        pois_,
    ], ignore_index=True)

    stop_ids   = [parada_sc_id]
    walk_times = [0.0]
    for _, poi in pois_.iterrows():
        cercanas = nearby_stops(poi["poi_lat"], poi["poi_lon"], stops, max_walk_min=15)
        if cercanas.empty:
            cercanas = nearby_stops(poi["poi_lat"], poi["poi_lon"], stops, max_walk_min=30)
        stop_ids.append(cercanas.iloc[0]["stop_id"])
        walk_times.append(cercanas.iloc[0]["walk_min"])

    nearby_per_poi = []
    for i in range(len(pois_con_origen)):
        poi = pois_con_origen.iloc[i]
        if i == 0:
            nearby_per_poi.append(pd.DataFrame({"stop_id": [stop_ids[0]], "walk_min": [0.0]}))
        else:
            cercanas = nearby_stops(poi["poi_lat"], poi["poi_lon"], stops, max_walk_min=15).head(4)
            if cercanas.empty:
                cercanas = pd.DataFrame({"stop_id": [stop_ids[i]], "walk_min": [walk_times[i]]})
            nearby_per_poi.append(cercanas[["stop_id", "walk_min"]])

    print(f"Origen: {name_origen}  |  tiempo_ida = {walk_hotel_to_pc_min:.1f} min")
    return walk_hotel_to_pc_min

Aquí vamos a elegir el día que haremos la ruta, hay que tener en cuenta que en el csv el formato está en días de la semana ["monday", "tuesday", ...] y en el gtfs estla de la siguiente forma "20260427"

In [ ]:
def setup_day(DATE, poi_time_df, calendar_dates, trips, stop_times):

  date = datetime.strptime(DATE, "%Y%m%d")
  weekdays = ["monday","tuesday","wednesday","thursday","friday","saturday","sunday"]
  weekday = weekdays[date.weekday()]

  open_pois_df = get_open_hours(poi_time_df, weekday) # Buscamos los pois abiertos ese día

  guaguas_availables = calendar_dates[
        (calendar_dates["date"] == int(DATE)) &
        (calendar_dates["exception_type"] == 1)
    ]["service_id"]
  trips_availables  = trips[trips["service_id"].isin(guaguas_availables)]["trip_id"]
  stop_times_day    = stop_times[stop_times["trip_id"].isin(trips_availables)]
  stop_times_day = stop_times_day.sort_values("stop_id").reset_index(drop=True)

  # Añadir los servicios del tranvía de ese día
  tram_services   = tram_calendar_dates(DATE)["service_id"]
  tram_trips_dia  = trips[trips["service_id"].isin(tram_services)]["trip_id"]
  stop_times_tram = stop_times[stop_times["trip_id"].isin(tram_trips_dia)]
  stop_times_day  = pd.concat([stop_times_day, stop_times_tram]) \
                      .drop_duplicates(subset=["trip_id","stop_id","stop_sequence"])

  if "departure_min" not in stop_times_day.columns:
        raise ValueError("stop_times no tiene 'departure_min' — ejecuta primero la celda de conversión de horarios")

  return weekday, open_pois_df, stop_times_day

Calculamos la matriz de distancia y tiempo

In [ ]:
def build_time_matrix(n_pois, dist_matrix, nearby_per_poi, stop_times_day,
                      init_hour, end_hour, franja_min=30,
                      velocidad_pie=4/60, max_walk_min=30):

    n_franjas = (end_hour - init_hour) // franja_min + 1
    time_matrix = np.full((n_franjas, n_pois, n_pois), np.inf)
    mode_matrix = np.full((n_franjas, n_pois, n_pois), "", dtype=object)
    bus_info = {}
    walk_matrix = dist_matrix / velocidad_pie

    # Paradas (stop_id, walk_min) de cada POI -> se sacan de de nearby_stops().head(4)
    paradas_por_poi = [
        list(zip(df["stop_id"].tolist(), df["walk_min"].tolist()))
        for df in nearby_per_poi
    ]

    # Caché alrededor de best_guagua_info: redondea la hora a la franja para aprovechar el mismo cálculo en horas parecidas.
    cache_guaguas = {}
    def cached_guagua(stop_o, stop_d, hora_salida):
        hora_slot = int(hora_salida // franja_min) * franja_min
        clave = (stop_o, stop_d, hora_slot)
        if clave in cache_guaguas:
            return cache_guaguas[clave]
        res = best_guagua_info(stop_o, stop_d, hora_salida, stop_times_day)
        cache_guaguas[clave] = res
        return res

    print(f"Construyendo matriz: {n_franjas} franjas × {n_pois} POIs")
    for t_idx in range(n_franjas):
        if t_idx % 5 == 0:
            print(f"  franja {t_idx}/{n_franjas}")
        t_min = init_hour + t_idx * franja_min

        for i in range(n_pois):
            time_matrix[t_idx, i, i] = 0
            mode_matrix[t_idx, i, i] = "mismo"
            for j in range(n_pois):
                if i == j:
                    continue
                t_pie = walk_matrix[i, j]

                mejor_total = np.inf
                mejor_info = None
                # probar las combinaciones de paradas cercanas de i y de j
                for stop_o, walk_o in paradas_por_poi[i]:
                    for stop_d, walk_d in paradas_por_poi[j]:
                        if stop_o == stop_d:
                            continue
                        info = cached_guagua(stop_o, stop_d, t_min + walk_o)
                        if info is None:
                            continue
                        espera = info["salida"]  - (t_min + walk_o)
                        viaje  = info["llegada"] - info["salida"]
                        total  = walk_o + espera + viaje + walk_d
                        if total < mejor_total:
                            mejor_total = total
                            mejor_info = {**info,
                                          "walk_o": walk_o, "walk_d": walk_d,
                                          "stop_o": stop_o,
                                          "stop_d": stop_d}

                UMBRAL_FORZAR_PIE = 12  # minutos

                if t_pie <= UMBRAL_FORZAR_PIE:
                    time_matrix[t_idx, i, j] = t_pie
                    mode_matrix[t_idx, i, j] = "pie"

                elif t_pie <= max_walk_min and t_pie <= mejor_total:
                    time_matrix[t_idx, i, j] = t_pie
                    mode_matrix[t_idx, i, j] = "pie"

                elif np.isfinite(mejor_total):
                    time_matrix[t_idx, i, j] = mejor_total
                    mode_matrix[t_idx, i, j] = "guagua"
                    bus_info[(t_idx, i, j)] = mejor_info

    return time_matrix, mode_matrix, bus_info

In [ ]:
def compute_intercity_leg(stop_times_day, init_hour):
    """Devuelve tiempo_ida. Si el hotel está dentro de la zona de POIs (HOTEL_CERCA),
       no hay guagua interurbana y el tiempo es solo la caminata a la parada más cercana."""
    if HOTEL_CERCA:
        # Hotel dentro de la zona: solo cuenta la caminata corta hasta la parada de inicio
        return walk_hotel_to_pc_min
    # Hotel lejos: guagua interurbana
    bus = time_guagua(parada_pc["stop_id"], parada_sc_id,
                      init_hour + walk_hotel_to_pc_min, stop_times_day)
    if np.isnan(bus):
        raise ValueError("No hay guagua interurbana ese día a esa hora")
    tiempo_ida = walk_hotel_to_pc_min + bus
    return tiempo_ida


def build_matrices(stop_times_day, init_urbano, max_urbano):
    """Devuelve dist_matrix, time_matrix, mode_matrix, bus_info."""
    n = len(pois_con_origen)
    coords = list(zip(pois_con_origen["poi_lat"], pois_con_origen["poi_lon"]))
    dist_matrix = build_distance_matrix(coords)
    time_matrix, mode_matrix, bus_info = build_time_matrix(
        n, dist_matrix, nearby_per_poi, stop_times_day,
        init_hour=init_urbano, end_hour=init_urbano + max_urbano)
    return dist_matrix, time_matrix, mode_matrix, bus_info


def run_grasp(n_iteraciones, time_matrix, init_urbano, max_urbano,
              candidate_pois, scores, verbose=True, required_pois=None):
    """Corre GRASP n veces y devuelve (mejor_ruta, mejor_score)."""
    best_route, best_score = [], 0
    for it in range(n_iteraciones):
        route = construct_route_grasp(
            POIs=pois_con_origen, travel_time=time_matrix,
            init_time=init_urbano, max_time=max_urbano,
            max_per_category=MAX_PER_CATEGORY,
            candidate_pois=candidate_pois,
            init_hour=init_urbano, rcl_size=5,
            required_pois=required_pois)
        score = compute_route_score(route, scores)
        if verbose:
            ids = "[" + ", ".join(str(i) for i in route) + "]"
            print(f"  iter {it+1:3d}: ruta={ids}  score={score:.1f}")
        if score > best_score:
            best_route, best_score = route, score
    return best_route, best_score


def run_vns_basic(ruta, time_matrix, init_urbano, max_urbano,
            candidate_pois, scores, k_max=5, required_pois=None):
    """Mejora una ruta con VNS básico. Devuelve (mejor_ruta, mejor_score)."""
    ruta = improve_vns_basic(
        route=ruta, POIs=pois_con_origen, travel_time=time_matrix,
        init_time=init_urbano, max_time=max_urbano,
        max_per_category=MAX_PER_CATEGORY,
        candidate_pois=candidate_pois, k_max=k_max,
        rcl_size=3, init_hour=init_urbano, franja_min=30,
        required_pois=required_pois)
    return ruta, compute_route_score(ruta, scores)

def run_vns(ruta, time_matrix, init_urbano, max_urbano,
                  candidate_pois, scores, k_max=5, repetitions=3,
                  required_pois=None):
    """Mejora una ruta con VNS extendido. Devuelve (mejor_ruta, mejor_score)."""
    ruta = improve_vns(
        route=ruta, POIs=pois_con_origen, travel_time=time_matrix,
        init_time=init_urbano, max_time=max_urbano,
        max_per_category=MAX_PER_CATEGORY,
        candidate_pois=candidate_pois, k_max=k_max,
        repetitions=repetitions,
        rcl_size=3, init_hour=init_urbano, franja_min=30,
        required_pois=required_pois)
    return ruta, compute_route_score(ruta, scores)


def plan_route(time_matrix, init_urbano, max_urbano, candidate_pois,
                    scores, n_grasp=20, seed=29):
    """Optimiza la ruta (GRASP + VNS extendido) usando una matriz ya construida."""
    random.seed(seed)
    ruta, score_grasp = run_grasp(n_grasp, time_matrix, init_urbano,
                                       max_urbano, candidate_pois, scores)
    print(f"Mejor ruta tras GRASP: score = {score_grasp}")
    random.seed(seed)
    ruta, score_vns = run_vns(ruta, time_matrix, init_urbano,
                                   max_urbano, candidate_pois, scores)
    print(f"Ruta tras VNS:         score = {score_vns}")

    if score_vns > score_grasp:
        print(f"El VNS MEJORÓ la ruta en {score_vns - score_grasp:.1f} puntos")
    else:
        print("El VNS NO mejoró sobre el GRASP")

    return ruta

def print_route(ruta, time_matrix, init_urbano, max_urbano, tiempo_ida=None):
    """Imprime la ruta como IDs y la lista de puntos visitados."""

    ids = [0] + list(ruta) + [0]
    print("Ruta:", ", ".join(str(i) for i in ids))

    print("\nPuntos que se visitan:")
    print(f"  {0}  Origen")
    for idx in ruta:
        name = pois_con_origen.iloc[idx]["poi_name"]
        print(f"  {idx}  {name}")

    visit_times = pois_con_origen["poi_visit_time"].astype(float).values
    dur = compute_route_time(ruta, init_urbano, time_matrix, visit_times,
                             initio=0, final=0,
                             init_hour=init_urbano, franja_min=30)
    print(f"\nDuración en Santa Cruz: {dur:.0f} min ({dur/60:.1f} h)")
    if tiempo_ida is not None:
        total = dur + 2 * tiempo_ida
        print(f"Con ida + vuelta al hotel: {total:.0f} min ({total/60:.1f} h)")

def print_route_schedule(ruta, time_matrix, init_urbano, franja_min=30):
    """Imprime la hora de llegada a cada POI y si está abierto en ese momento."""
    n_franja = time_matrix.shape[0]
    t = init_urbano
    anterior = 0
    todo_ok = True
    for nodo in ruta:
        franja = int(np.clip((t - init_urbano) // franja_min, 0, n_franja - 1))
        t += time_matrix[franja, anterior, nodo]
        abierto = is_open(nodo, t, ventanas_horario)
        if not abierto:
            todo_ok = False
        ventanas = ventanas_horario.get(nodo)
        ventanas_txt = ("siempre abierto / sin horario" if not ventanas else
                        ", ".join(f"{a//60:02d}:{a%60:02d}-{c//60:02d}:{c%60:02d}"
                                  for a, c in ventanas))
        marca = "OK  " if abierto else "CERRADO <--"
        print(f"{marca} {int(t)//60:02d}:{int(t)%60:02d}  "
              f"{pois_con_origen.iloc[nodo]['poi_name'][:42]:43s} [{ventanas_txt}]")
        t += visit_times[nodo]
        anterior = nodo

    estado = "Todos los POIs abiertos a su hora de llegada." if todo_ok \
             else "HAY POIs CERRADOS en la ruta — revisar."
    print(f"\n{estado}")
    return todo_ok

Patrón estrategia para la mejora de rutas

In [ ]:
# ============================================================
# PATRON STRATEGY
# ============================================================
class ImprovementStrategy(ABC):
    """
    Interfaz comun de las estrategias de mejora de rutas (patron Strategy).

    Cada estrategia concreta encapsula una metaheuristica distinta. El codigo
    cliente trabaja contra esta interfaz, sin conocer que algoritmo hay detras,
    y puede intercambiarlos en tiempo de ejecucion.
    """
    @abstractmethod
    def improve(self, ruta, time_matrix, init_urbano, max_urbano, candidate_pois,
                scores, required_pois=None):
        """Mejora una ruta y devuelve (ruta_mejorada, score)."""


    @property
    @abstractmethod
    def name(self):
        """Nombre legible de la estrategia."""



class VNSStrategy(ImprovementStrategy):
    name = "VNS extendido"
    def improve(self, ruta, time_matrix, init_urbano, max_urbano, candidate_pois,
                scores, required_pois=None):
        return run_vns(ruta, time_matrix, init_urbano, max_urbano,
                             candidate_pois, scores, required_pois=required_pois)

class BASICVNSStrategy(ImprovementStrategy):
    name = "VNS básico"
    def improve(self, ruta, time_matrix, init_urbano, max_urbano, candidate_pois,
                scores, required_pois=None):
        return run_vns_basic(ruta, time_matrix, init_urbano, max_urbano,
                      candidate_pois, scores, required_pois=required_pois)

class NoImprovement(ImprovementStrategy):
    name = "Solo GRASP"
    def improve(self, ruta, time_matrix, init_urbano, max_urbano, candidate_pois,
                scores, required_pois=None):
        return ruta, compute_route_score(ruta, scores)


Patrón fachada

In [ ]:
#================================================================================================================
# Patrón Fachada clase RouteRecommender
#================================================================================================================
class RouteRecommender:
    """
    Fachada del sistema recomendador (patrón Facade).

    Separa la preparación del día (lenta: matriz de tiempos) de la generación
    de la ruta (rápida). La matriz se cachea y se reutiliza mientras no cambie
    el día, el hotel o el municipio.
    """

    def __init__(self, estrategia=None):
        self.estrategia = estrategia or BASICVNSStrategy()
        self._cache = {}          # datos del día preparado

    def set_strategy(self, estrategia):
        self.estrategia = estrategia
        return self

    def prepare_day(self, dia, horas=7, municipio="Santa Cruz", hotel=None,
                     verbose=True):
        """
        Prepara TODO lo costoso de un día: hotel, transporte y matriz de tiempos.
        Guarda el resultado en caché para reutilizarlo en sucesivas llamadas a
        generate() con los mismos día/hotel/municipio.
        """
        clave = (dia, hotel, municipio, horas)
        if clave in self._cache:
            if verbose:
                print("(Reutilizando la matriz ya calculada para este día/hotel/municipio)")
            return self._cache[clave]

        if hotel is not None:
            prepare_hotel(hotel, municipio)

        DATE = dia[4:8] + dia[2:4] + dia[0:2]
        init_hour = 9 * 60
        max_time = horas * 60

        weekday, open_pois_df, stop_times_day = setup_day(
            DATE, poi_time_df, calendar, trips, stop_times)
        tiempo_ida = compute_intercity_leg(stop_times_day, init_hour)
        max_urbano = int(max_time - 2 * tiempo_ida)
        init_urbano = int(init_hour + tiempo_ida)

        dist_matrix, time_matrix, mode_matrix, bus_info = build_matrices(
            stop_times_day, init_urbano, max_urbano)

        global ventanas_horario, scores, visit_times, category, MAX_PER_CATEGORY
        scores = pois_con_origen["poi_score"].astype(float).values
        visit_times = pois_con_origen["poi_visit_time"].astype(float).values
        category = dict(zip(pois_con_origen.index, pois_con_origen["poi_cat"]))
        MAX_PER_CATEGORY = {
            "park": 10, "museum": 10, "nature_preserve": 10,
            "hiking_area": 10, "garden": 10, "scenic_spot": 10,
            "tourist_attraction": 10, "historical_landmark": 10,
            "mountain_peak": 10,
        }
        ventanas_poi = build_time_windows(open_pois_df)
        ventanas_horario = {idx: ventanas_poi.get(pois_con_origen.iloc[idx]["poi_id"])
                            for idx in range(len(pois_con_origen))}
        open_pois = open_pois_df["poi_id"].unique().tolist()

        if municipio == "Santa Cruz":
            municipios_ok = {"Santa Cruz"}
        elif municipio == "La Laguna":
            municipios_ok = {"La Laguna"}
        else:
            municipios_ok = {"Santa Cruz", "La Laguna"}

        candidate_pois_base = [
            i for i in range(1, len(pois_con_origen))
            if pois_con_origen.iloc[i]["poi_id"] in open_pois
            and pois_con_origen.iloc[i].get("municipio", "Otro") in municipios_ok
        ]

        # Guardamos en caché todo lo costoso del día
        self._cache[clave] = {
            "dia": dia, "horas": horas, "municipio": municipio,
            "time_matrix": time_matrix, "mode_matrix": mode_matrix,
            "bus_info": bus_info, "init_urbano": init_urbano,
            "max_urbano": max_urbano, "tiempo_ida": tiempo_ida,
            "stop_times_day": stop_times_day,
            "candidate_pois_base": candidate_pois_base,
            "ventanas_horario": ventanas_horario, "scores": scores,
        }
        return self._cache[clave]

    def generate(self, dia, horas=7, municipio="Santa Cruz", hotel=None,
                n_grasp=50, semilla=29, excluir=None, incluir=None,
                candidatos_previos=None, verbose=True):
        """Genera la ruta de UN día. Reutiliza la matriz cacheada si coincide día/hotel/municipio."""
        datos = self.prepare_day(dia, horas=horas, municipio=municipio,
                                  hotel=hotel, verbose=verbose)

        time_matrix = datos["time_matrix"]
        mode_matrix = datos["mode_matrix"]
        bus_info    = datos["bus_info"]
        init_urbano = datos["init_urbano"]
        max_urbano  = datos["max_urbano"]
        tiempo_ida  = datos["tiempo_ida"]
        scores      = datos["scores"]

        if candidatos_previos is not None:
            candidate_pois = list(candidatos_previos)
        else:
            candidate_pois = list(datos["candidate_pois_base"])

        if excluir:
            for name in excluir:
                candidate_pois = remove_poi(name, candidate_pois)

        requested_pois = resolve_pois(incluir)
        required_pois = [p for p in requested_pois if p in candidate_pois]
        unavailable_required = [p for p in requested_pois if p not in candidate_pois]
        if verbose and unavailable_required:
            names_no_disp = [pois_con_origen.iloc[i]["poi_name"] for i in unavailable_required]
            print("No se pueden incluir porque no están disponibles con la fecha/municipio/filtros actuales: "
                  + ", ".join(names_no_disp))

        if verbose:
            print(f"Fecha: {dia[:2]}/{dia[2:4]}/{dia[4:]}  |  {horas} h  |  {municipio}")
            print(f"Estrategia de mejora: {self.estrategia.name}  |  GRASP n={n_grasp}")
            print(f"POIs candidatos: {len(candidate_pois)}")
            if excluir:
                print(f"Excluidos: {', '.join(excluir)}")
            if required_pois:
                names_req = [pois_con_origen.iloc[i]["poi_name"] for i in required_pois]
                print(f"POIs obligatorios: {', '.join(names_req)}")

        if not candidate_pois:
            if verbose:
                print("No hay POIs disponibles ese día con esos filtros.")
            return None

        random.seed(semilla)
        ruta, score_grasp = run_grasp(n_grasp, time_matrix, init_urbano, max_urbano,
                                      candidate_pois, scores, verbose=verbose,
                                      required_pois=required_pois)
        random.seed(semilla)
        ruta, score_mejora = self.estrategia.improve(
            ruta, time_matrix, init_urbano, max_urbano, candidate_pois, scores,
            required_pois=required_pois)

        if verbose:
            print(f"\nScore tras GRASP:       {score_grasp:.1f}")
            print(f"Score tras {self.estrategia.name}: {score_mejora:.1f}")
            print("-" * 55)
            print_route(ruta, time_matrix, init_urbano, max_urbano, tiempo_ida)

        return {
            "ruta": ruta, "score_grasp": score_grasp, "score_mejora": score_mejora,
            "estrategia": self.estrategia.name, "time_matrix": time_matrix,
            "mode_matrix": mode_matrix, "bus_info": bus_info,
            "init_urbano": init_urbano, "max_urbano": max_urbano,
            "tiempo_ida": tiempo_ida, "stop_times_day": datos["stop_times_day"],
            "candidate_pois": candidate_pois, "required_pois": required_pois,
        }
    def replan_without(self, res_previo, excluir, rcl_size=3):
        """Quita los POIs indicados de una ruta ya generada y repara el hueco con GRASP."""
        fuera = resolve_pois(excluir)

        ruta_base = [
            p for p in res_previo["ruta"]
            if p not in fuera
        ]

        candidate_pois = [
            p for p in res_previo["candidate_pois"]
            if p not in fuera
        ]

        candidatos = [
            p for p in candidate_pois
            if p not in ruta_base
        ]

        ruta_nueva = construct_route_grasp(
            POIs=pois_con_origen,
            travel_time=res_previo["time_matrix"],
            init_time=res_previo["init_urbano"],
            max_time=res_previo["max_urbano"],
            max_per_category=MAX_PER_CATEGORY,
            candidate_pois=candidatos,
            init_hour=res_previo["init_urbano"],
            rcl_size=rcl_size,
            route=ruta_base,
        )

        resultado = dict(res_previo)
        resultado["ruta"] = ruta_nueva
        resultado["candidate_pois"] = candidate_pois
        resultado["score_mejora"] = compute_route_score(ruta_nueva, scores)
        resultado["excluidos"] = excluir

        return resultado

    def replan_with(self, res_previo, incluir, rcl_size=3):
        """Añade POIs obligatorios a una ruta ya generada y repara el hueco con GRASP."""
        pedidos = resolve_pois(incluir)
        ruta_base = list(res_previo["ruta"])

        no_disponibles = [
            p for p in pedidos
            if p not in res_previo["candidate_pois"]
        ]

        required_pois = [
            p for p in pedidos
            if p in res_previo["candidate_pois"]
        ]

        if no_disponibles:
            nombres = [pois_con_origen.iloc[p]["poi_name"] for p in no_disponibles]
            print("No se pueden añadir porque no están disponibles:", ", ".join(nombres))

        candidatos = [
            p for p in res_previo["candidate_pois"]
            if p not in ruta_base
        ]

        ruta_nueva = construct_route_grasp(
            POIs=pois_con_origen,
            travel_time=res_previo["time_matrix"],
            init_time=res_previo["init_urbano"],
            max_time=res_previo["max_urbano"],
            max_per_category=MAX_PER_CATEGORY,
            candidate_pois=candidatos,
            init_hour=res_previo["init_urbano"],
            rcl_size=rcl_size,
            route=ruta_base,
            required_pois=required_pois,
        )

        resultado = dict(res_previo)
        resultado["ruta"] = ruta_nueva
        resultado["score_mejora"] = compute_route_score(ruta_nueva, scores)
        resultado["required_pois"] = required_pois

        return resultado

    def generate_stay(self, dia_inicio, num_dias=None, horas=7, municipio="Santa Cruz",
                         hotel=None, n_grasp=50, semilla=29, excluir=None, verbose=True):
        """
        Planifica VARIOS días repartiendo los POIs sin repetirlos.

        horas puede ser:
          - un número: las mismas horas todos los días (p.ej. horas=7)
          - una lista: horas distintas por día (p.ej. horas=[5, 8, 8, 4]).
            En ese caso num_dias se deduce de la longitud de la lista.

        Cada día se calcula en silencio y solo se imprime si consigue una ruta con
        POIs; si no, se detiene la estancia con un mensaje (sin volcar iteraciones vacías).
        """
        # Normalizamos horas a una lista, una entrada por día
        if isinstance(horas, (list, tuple)):
            horas_por_dia = list(horas)
            num_dias = len(horas_por_dia)
        else:
            if num_dias is None:
                raise ValueError("Indica num_dias o pasa una lista de horas.")
            horas_por_dia = [horas] * num_dias

        if hotel is not None:
            prepare_hotel(hotel, municipio)

        if municipio == "Santa Cruz":
            municipios_ok = {"Santa Cruz"}
        elif municipio == "La Laguna":
            municipios_ok = {"La Laguna"}
        else:
            municipios_ok = {"Santa Cruz", "La Laguna"}
        disponibles = [
            i for i in range(1, len(pois_con_origen))
            if pois_con_origen.iloc[i].get("municipio", "Otro") in municipios_ok
        ]

        d0 = datetime.strptime(dia_inicio, "%d%m%Y")
        resultados = []
        for n in range(num_dias):
            dia = (d0 + timedelta(days=n)).strftime("%d%m%Y")
            horas_hoy = horas_por_dia[n]

            if verbose:
                fecha_bonita = f"{dia[:2]}/{dia[2:4]}/{dia[4:]}"
                print("=" * 55)
                print(f"DÍA {n+1} de {num_dias}  -  {fecha_bonita}  ({horas_hoy} h)")
                print("=" * 55)
                print(f"POIs disponibles para hoy: {len(disponibles)}")

            if not disponibles:
                if verbose:
                    print("Ya no quedan POIs por visitar. Se detiene la estancia.")
                break

            # Calculamos el día sin volcar las iteraciones
            res = self.generate(dia, horas=horas_hoy, municipio=municipio, hotel=None,
                              n_grasp=n_grasp, semilla=semilla, excluir=excluir,
                              candidatos_previos=disponibles, verbose=False)

            visitados = [p for p in res["ruta"] if p != 0] if res else []

            # Si el día no coloca ningún POI, paramos sin imprimir iteraciones vacías
            if not visitados:
                if verbose:
                    print("\nNo quedan POIs que se puedan visitar con estas condiciones.")
                    print("(Puede que los POIs restantes estén en otro municipio y no")
                    print(" dé tiempo a llegar con las horas indicadas. Prueba más horas.)")
                    print("Se detiene la planificación de la estancia.")
                break

            # El día tiene ruta: lo imprimimos ahora
            if verbose:
                print(f"Score: {res['score_mejora']:.1f}  ({res['estrategia']})")
                print("-" * 55)
                print_route(res["ruta"], res["time_matrix"], res["init_urbano"],
                           res["max_urbano"], res["tiempo_ida"])

            disponibles = [p for p in disponibles if p not in visitados]
            res["dia"] = dia
            res["num_dia"] = n + 1
            res["horas"] = horas_hoy
            resultados.append(res)

            if not disponibles:
                if verbose:
                    print("\n" + "=" * 55)
                    print(f"Todos los POIs han sido visitados en {n+1} día(s).")
                    print("=" * 55)
                break

        return resultados


def run_grasp_checkpoints(checkpoints, time_matrix, init_urbano, max_urbano,
                          candidate_pois, scores, required_pois=None):
    """Ejecuta max(checkpoints) GRASP y guarda el mejor resultado en cada n."""
    best_route, best_score = [], 0
    snapshots = {}
    t0 = time.perf_counter()

    for it in range(1, max(checkpoints) + 1):
        route = construct_route_grasp(
            POIs=pois_con_origen,
            travel_time=time_matrix,
            init_time=init_urbano,
            max_time=max_urbano,
            max_per_category=MAX_PER_CATEGORY,
            candidate_pois=candidate_pois,
            init_hour=init_urbano,
            rcl_size=5,
            required_pois=required_pois,
        )
        score = compute_route_score(route, scores)
        if score > best_score:
            best_route, best_score = route, score

        if it in checkpoints:
            snapshots[it] = {
                "ruta": list(best_route),
                "score": best_score,
                "t_grasp_s": time.perf_counter() - t0,
            }

    return snapshots

Preparación de mapas

In [ ]:
#================================================================================================================
# Visualización de mapas con Folium
#================================================================================================================

osrm_cache = {}

def osrm_route(coords_lonlat, perfil="foot"):
    """coordenadas_lonlat: lista de [lon, lat]. Devuelve [(lat, lon), ...]."""
    clave = (perfil, tuple(tuple(c) for c in coords_lonlat))
    if clave in osrm_cache:
        return osrm_cache[clave]
    coord_str = ";".join(f"{lon},{lat}" for lon, lat in coords_lonlat)
    base = "routed-foot/route/v1/foot" if perfil == "foot" else "routed-car/route/v1/driving"
    url = f"https://routing.openstreetmap.de/{base}/{coord_str}"
    resp = requests.get(url, params={"overview": "full", "geometries": "geojson"},
                        headers={"User-Agent": "tfg-notebook/1.0"})
    resp.raise_for_status()
    geom = resp.json()["routes"][0]["geometry"]["coordinates"]
    pts = [(lat, lon) for lon, lat in geom]
    osrm_cache[clave] = pts
    return pts

def safe_line(coord_o_lonlat, coord_d_lonlat, perfil="foot"):
    """OSRM con fallback a línea recta si falla."""
    try:
        return osrm_route([coord_o_lonlat, coord_d_lonlat], perfil=perfil)
    except Exception:
        return [(coord_o_lonlat[1], coord_o_lonlat[0]),
                (coord_d_lonlat[1], coord_d_lonlat[0])]

def trip_stops(trip_id, seq_o, seq_d):
    paradas = stop_times_day[
        (stop_times_day["trip_id"] == trip_id) &
        (stop_times_day["stop_sequence"] >= seq_o) &
        (stop_times_day["stop_sequence"] <= seq_d)
    ].sort_values("stop_sequence")
    paradas = paradas.merge(stops[["stop_id", "stop_lat", "stop_lon", "stop_name"]], on="stop_id")
    info_linea = trips[trips["trip_id"] == trip_id].merge(routes, on="route_id").iloc[0]
    return paradas, info_linea

def draw(r, titulo="Ruta"):
    # Si no hay resultado o la ruta no tiene POIs, no dibujamos
    if r is None or not [p for p in r["ruta"] if p != 0]:
        print("No hay ruta que dibujar: no se encontraron POIs disponibles con estas condiciones.")
        return None
    global stop_times_day
    stop_times_day = r["stop_times_day"]
    return draw_map(r["ruta"], r["time_matrix"], r["mode_matrix"],
                    r["bus_info"], r["init_urbano"], titulo=titulo)
# Función principal
def draw_map(ruta, time_matrix, mode_matrix, bus_info,
                 init_urbano, franja_min=30, titulo=""):
    """Dibuja la ruta. Si el hotel está lejos, añade la etapa interurbana."""
    n_franjas = time_matrix.shape[0]

    def to_franja(t):
        if not np.isfinite(t):
            return n_franjas - 1
        return int(np.clip((t - init_urbano) // franja_min, 0, n_franjas - 1))

    ruta_completa_idx = [0] + list(ruta) + [0]

    lats = [HOTEL_LAT] + [pois_con_origen.iloc[i]["poi_lat"] for i in ruta_completa_idx]
    lons = [HOTEL_LON] + [pois_con_origen.iloc[i]["poi_lon"] for i in ruta_completa_idx]
    m = folium.Map(location=[sum(lats)/len(lats), sum(lons)/len(lons)],
                   zoom_start=12, tiles="CartoDB positron")

    capa_pie     = folium.FeatureGroup(name="A pie")
    capa_bus     = folium.FeatureGroup(name="Guagua")
    capa_tranvia = folium.FeatureGroup(name="Tranvía")
    capa_inter   = folium.FeatureGroup(name="Guagua interurbana (hotel)")
    capa_paradas = folium.FeatureGroup(name="Paradas")
    capa_pois    = folium.FeatureGroup(name="POIs")

    coord_hotel = [HOTEL_LON, HOTEL_LAT]
    coord_sc    = [pois_con_origen.iloc[0]["poi_lon"], pois_con_origen.iloc[0]["poi_lat"]]  # origen

    # etapa interurbana: solo si el hotel está LEJOS
    if not HOTEL_CERCA:
        coord_pc = [parada_pc["stop_lon"], parada_pc["stop_lat"]]
        # IDA: hotel -> parada PC (a pie)
        geom = safe_line(coord_hotel, coord_pc, "foot")
        folium.PolyLine(geom, color="#1971c2", weight=5, opacity=0.85,
            tooltip=f"A pie hotel → {parada_pc['stop_name']} ({walk_hotel_to_pc_min:.0f} min)").add_to(capa_pie)
        # IDA: guagua interurbana PC -> SC
        geom = safe_line(coord_pc, coord_sc, "car")
        folium.PolyLine(geom, color="#9c36b5", weight=6, opacity=0.9,
            tooltip=f"Guagua interurbana → {parada_sc['stop_name']}").add_to(capa_inter)
        folium.Marker([parada_pc["stop_lat"], parada_pc["stop_lon"]],
            popup=f"Subida interurbana: {parada_pc['stop_name']}",
            icon=folium.Icon(color="purple", icon="bus", prefix="fa")).add_to(capa_paradas)
    else:
        # Hotel cerca: camino del hotel al primer Poi/origen
        geom = safe_line(coord_hotel, coord_sc, "foot")
        folium.PolyLine(geom, color="#1971c2", weight=5, opacity=0.85,
            tooltip="A pie desde el hotel").add_to(capa_pie)

    # Marcador del hotel
    folium.Marker([HOTEL_LAT, HOTEL_LON], popup="Hotel",
        icon=folium.Icon(color="darkred", icon="home", prefix="fa")).add_to(capa_pois)

    # Ruta Urbana: origen -> POIs -> origen
    t_actual = init_urbano
    resumen = []
    for k in range(len(ruta_completa_idx) - 1):
        i, j = ruta_completa_idx[k], ruta_completa_idx[k+1]
        if i == j:
            continue
        poi_i, poi_j = pois_con_origen.iloc[i], pois_con_origen.iloc[j]
        coord_i = [poi_i["poi_lon"], poi_i["poi_lat"]]
        coord_j = [poi_j["poi_lon"], poi_j["poi_lat"]]
        name_i, name_j = poi_i["poi_name"], poi_j["poi_name"]

        franja = to_franja(t_actual)
        modo   = mode_matrix[franja, i, j]
        tiempo = time_matrix[franja, i, j]

        if modo == "pie":
            geom = safe_line(coord_i, coord_j, "foot")
            folium.PolyLine(geom, color="#1971c2", weight=5, opacity=0.85,
                tooltip=f"A pie: {name_i} → {name_j} ({tiempo:.0f} min)").add_to(capa_pie)
            resumen.append(f"{k+1}. A pie {name_i} → {name_j} ({tiempo:.0f} min)")

        elif modo == "guagua":
            info = bus_info[(franja, i, j)]
            paradas, linea = trip_stops(info["trip_id"], info["seq_o"], info["seq_d"])

            es_tranvia  = ("route_type" in linea.index and linea["route_type"] == 0)
            name_modo = "Tranvía" if es_tranvia else "Guagua"
            color_modo  = "#2f9e44" if es_tranvia else "#e8590c"
            capa_modo   = capa_tranvia if es_tranvia else capa_bus
            name_linea = f"L{linea['route_short_name']} - {linea['route_long_name']}"

            sub, baj = paradas.iloc[0], paradas.iloc[-1]
            coord_sub = [sub["stop_lon"], sub["stop_lat"]]
            coord_baj = [baj["stop_lon"], baj["stop_lat"]]

            folium.PolyLine(safe_line(coord_i, coord_sub, "foot"),
                color="#1971c2", weight=5, opacity=0.85,
                tooltip=f"A pie a {sub['stop_name']}").add_to(capa_pie)
            coords_transp = [[r["stop_lon"], r["stop_lat"]] for _, r in paradas.iterrows()]
            try:
                geom_b = osrm_route(coords_transp, "car")
            except requests.RequestException:
                geom_b = [(r["stop_lat"], r["stop_lon"]) for _, r in paradas.iterrows()]
            folium.PolyLine(geom_b, color=color_modo, weight=6, opacity=0.9,
                tooltip=f"{name_modo} {name_linea}").add_to(capa_modo)
            folium.PolyLine(safe_line(coord_baj, coord_j, "foot"),
                color="#1971c2", weight=5, opacity=0.85,
                tooltip=f"A pie desde {baj['stop_name']}").add_to(capa_pie)

            icono = "train" if es_tranvia else "bus"
            color_parada = "green" if es_tranvia else "orange"
            folium.Marker([sub["stop_lat"], sub["stop_lon"]],
                popup=f"Subida ({name_modo} {name_linea}): {sub['stop_name']}",
                icon=folium.Icon(color=color_parada, icon=icono, prefix="fa")).add_to(capa_paradas)
            folium.Marker([baj["stop_lat"], baj["stop_lon"]],
                popup=f"Bajada: {baj['stop_name']}",
                icon=folium.Icon(color=color_parada, icon=icono, prefix="fa")).add_to(capa_paradas)

            resumen.append(f"{k+1}. {name_modo} {name_linea}: {name_i} → {name_j} ({tiempo:.0f} min)")
        else:
            resumen.append(f"{k+1}. {name_i} → {name_j}: sin conexión")

        if np.isfinite(tiempo):
            t_actual += tiempo
        if j != 0:
            t_actual += visit_times[j]

    # vuelta -> solo si el hotel está lejos
    if not HOTEL_CERCA:
        coord_pc = [parada_pc["stop_lon"], parada_pc["stop_lat"]]
        geom = safe_line(coord_sc, coord_pc, "car")
        folium.PolyLine(geom, color="#9c36b5", weight=6, opacity=0.9, dash_array="1,6",
            tooltip="Guagua interurbana de vuelta").add_to(capa_inter)
        geom = safe_line(coord_pc, coord_hotel, "foot")
        folium.PolyLine(geom, color="#1971c2", weight=5, opacity=0.85,
            tooltip="A pie de vuelta al hotel").add_to(capa_pie)
    else:
        geom = safe_line(coord_sc, coord_hotel, "foot")
        folium.PolyLine(geom, color="#1971c2", weight=5, opacity=0.85,
            tooltip="A pie de vuelta al hotel").add_to(capa_pie)

    # Marcadores de los pois
    for orden, idx in enumerate(ruta_completa_idx):
        poi = pois_con_origen.iloc[idx]
        extremo = (orden == 0 or orden == len(ruta_completa_idx) - 1)
        color = "red" if extremo else "blue"
        etiqueta = (f"Inicio/Fin: {poi['poi_name']}" if extremo
                    else f"Parada {orden}: {poi['poi_name']}")
        folium.Marker([poi["poi_lat"], poi["poi_lon"]],
            popup=etiqueta, tooltip=etiqueta,
            icon=folium.Icon(color=color, icon="info-sign")).add_to(capa_pois)

    for capa in (capa_inter, capa_bus, capa_tranvia, capa_pie, capa_paradas, capa_pois):
        capa.add_to(m)
    folium.LayerControl(collapsed=False).add_to(m)
    m.fit_bounds([[min(lats), min(lons)], [max(lats), max(lons)]])

    print(f"\n=== Resumen ruta {titulo} ===")
    for ln in resumen:
        print(ln)
    return m

Código para modificar la ruta, eliminar poi y añadir poi (en los csv)

In [ ]:
def resolve_pois(pois_a_incluir):
    """Devuelve los índices de los POIs indicados por nombre parcial o por índice."""
    if pois_a_incluir is None:
        return []
    if not isinstance(pois_a_incluir, (list, tuple, set)):
        pois_a_incluir = [pois_a_incluir]

    indices = []
    for poi in pois_a_incluir:
        if isinstance(poi, str):
            coincidencias = pois_con_origen.index[
                pois_con_origen["poi_name"].str.contains(poi, case=False, na=False)]
            if len(coincidencias) == 0:
                print(f"No encontré ningún POI que contenga '{poi}'")
                continue
            idx = int(coincidencias[0])
        else:
            idx = int(poi)

        if idx == 0:
            print("El índice 0 corresponde al origen/hotel, no a un POI visitable.")
            continue
        if idx not in indices:
            indices.append(idx)
    return indices


def remove_poi(pois_a_quitar, candidate_pois):
    """Devuelve candidate_pois sin los POIs indicados (por nombre o índice)."""
    indices_fuera = resolve_pois(pois_a_quitar)
    for idx in indices_fuera:
        print(f"Quitando: {pois_con_origen.iloc[idx]['poi_name']} (índice {idx})")
    return [i for i in candidate_pois if i not in indices_fuera]


In [ ]:
#================================================================================================================
# Diccionario de Hoteles (lat, lon)
#================================================================================================================
HOTELES = {
    "Hotel Nivaria (La Laguna)":          (28.486773940163136, -16.313186397074528),
    "Hotel Atlántico Centro (Santa Cruz)":  (28.4670837389, -16.2497802193),
    "Hotel Barceló Contemporáneo (Santa Cruz)": (28.4739846389, -16.2528175094),
    "Bahía Príncipe (Puerto de la Cruz)": (28.416272, -16.54143)
}

Generamos las rutas

In [ ]:
# ==================================================================
# Generar rutas con distinto número de construcciones GRASP,
# usando la Facade (RouteRecommender) y el Strategy (VNS)
# ==================================================================
fecha     = "16072026"
horas     = 10
municipio = "Ambos"
hotel     = "Hotel Atlántico Centro (Santa Cruz)"
semilla   = 29
CHECKPOINTS = [20, 50, 100, 150]

reco = RouteRecommender(VNSStrategy())

datos = reco.prepare_day(fecha, horas=horas, municipio=municipio,
                         hotel=hotel, verbose=True)
time_matrix    = datos["time_matrix"]
init_urbano    = datos["init_urbano"]
max_urbano     = datos["max_urbano"]
scores         = datos["scores"]
candidate_pois = datos["candidate_pois_base"]

random.seed(semilla)
snaps = run_grasp_checkpoints(CHECKPOINTS, time_matrix, init_urbano,
                              max_urbano, candidate_pois, scores)

resultados = {}
filas = []
for n in CHECKPOINTS:
    print(f"===== {n} EJECUCIONES =====")
    t0 = time.perf_counter()
    random.seed(semilla)
    ruta_vns, score_vns = reco.estrategia.improve(
        snaps[n]["ruta"], time_matrix, init_urbano, max_urbano,
        candidate_pois, scores,
    )
    t_vns_s = time.perf_counter() - t0

    resultados[n] = {
        "ruta": ruta_vns,
        "score_grasp": snaps[n]["score"],
        "score_mejora": score_vns,
        "estrategia": reco.estrategia.name,
        "time_matrix": time_matrix,
        "mode_matrix": datos["mode_matrix"],
        "bus_info": datos["bus_info"],
        "stop_times_day": datos["stop_times_day"],
        "init_urbano": init_urbano,
        "max_urbano": max_urbano,
        "tiempo_ida": datos["tiempo_ida"],
        "candidate_pois": candidate_pois,
    }
    filas.append({
        "n_grasp": n,
        "score_grasp": snaps[n]["score"],
        "score_mejora": score_vns,
        "t_grasp_s": snaps[n]["t_grasp_s"],
        "t_vns_s": t_vns_s,
        "t_total_s": snaps[n]["t_grasp_s"] + t_vns_s,
    })
    print(f"GRASP={snaps[n]['score']:.1f}  {reco.estrategia.name}={score_vns:.1f}  "
          f"tiempo={filas[-1]['t_total_s']:.1f}s")
    print("-" * 55)
    print_route(ruta_vns, time_matrix, init_urbano, max_urbano,
                tiempo_ida=datos["tiempo_ida"])
    print()
df_resultados = pd.DataFrame(filas)
display(df_resultados)

Hotel seleccionado: Hotel Atlántico Centro (Santa Cruz)  (28.4670837389, -16.2497802193)
POI más cercano del municipio destino: 0.13 km
Hotel CERCA: se empieza caminando (sin guagua interurbana)
Origen = hotel. Parada más cercana: 9360
Origen: Hotel (Hotel Atlántico Centro (Santa Cruz))  |  tiempo_ida = 0.0 min
Construyendo matriz: 21 franjas × 38 POIs
  franja 0/21
  franja 5/21
  franja 10/21
  franja 15/21
  franja 20/21
===== 20 EJECUCIONES =====
GRASP=67.4  VNS=67.4  tiempo=8.7s
-------------------------------------------------------
Ruta: 0, 19, 1, 17, 20, 5, 2, 8, 15, 10, 21, 16, 11, 7, 14, 18, 0

Puntos que se visitan:
  0  Origen
  19  Plaza Príncipe de Asturias
  1  Parque García Sanabria
  17  Mirador los Campitos
  20  Plaza Weyler
  5  Iglesia Matriz de Nuestra Señora de la Concepción
  2  Mercado Ntra. Señora de África
  8  Casa del Carnaval
  15  Castillo San Juan Bautista
  10  Auditorio de Tenerife Adán Martín
  21  Calle Castillo
  16  Plaza del Chicharro
  11  Plaza 

,n_grasp,score_grasp,score_mejora,t_grasp_s,t_vns_s,t_total_s
0,20,67.4,67.4,4.751790,3.941655,8.693445
1,50,67.4,67.4,10.591412,3.355696,13.947108
2,100,71.6,84.5,21.218733,4.656642,25.875375
3,150,71.6,84.5,31.428096,5.235460,36.663556


In [ ]:
#================================================================================================================
# Comprobamos si cuando llega a los POI están abiertos
#================================================================================================================
print_route_schedule(resultados[100]["ruta"], time_matrix, init_urbano)

OK   09:02  Plaza del Chicharro                         [00:00-23:59]
OK   09:19  Plaza Príncipe de Asturias                  [00:00-23:59]
OK   09:41  Parroquia de San Francisco de Asís          [08:30-13:00, 17:30-20:00]
OK   10:11  Castillo de San Cristóbal                   [10:00-18:00]
OK   10:41  Plaza de España                             [00:00-23:59]
OK   11:30  Castillo San Juan Bautista                  [00:00-23:59]
OK   12:21  Iglesia Santo Domingo de Guzmán             [10:00-15:00, 18:00-20:00]
OK   12:54  Ermita de San Miguel                        [17:00-20:00, 11:00-14:00]
OK   13:14  Plaza del Cristo de La Laguna               [07:00-20:00]
OK   13:44  Casa Salazar                                [09:00-14:00]
OK   14:05  Calle San Agustín                           [00:00-23:59]
OK   14:28  Parroquia Matriz de Nuestra Señora de la C  [10:00-17:00]
OK   14:57  Calle Obispo Rey Redondo                    [00:00-23:59]
OK   15:16  Casa de los Capitanes Generales        

True

Visualización de los mapas


In [ ]:
r = resultados[20]
draw(resultados[20], titulo="Ruta (20 ejecuciones)")


=== Resumen ruta Ruta (20 ejecuciones) ===
1. A pie Hotel (Hotel Atlántico Centro (Santa Cruz)) → Plaza Príncipe de Asturias (3 min)
2. A pie Plaza Príncipe de Asturias → Parque García Sanabria (12 min)
3. A pie Parque García Sanabria → Mirador los Campitos (17 min)
4. A pie Mirador los Campitos → Plaza Weyler (21 min)
5. Tranvía LL1 - Linea 1: Plaza Weyler → Iglesia Matriz de Nuestra Señora de la Concepción (6 min)
6. A pie Iglesia Matriz de Nuestra Señora de la Concepción → Mercado Ntra. Señora de África (7 min)
7. A pie Mercado Ntra. Señora de África → Casa del Carnaval (9 min)
8. Guagua L919 - INTERCAMBIADOR BARRIO SALUD CUESTA PIEDRA  POR BARRANCO DE SANTOS: Casa del Carnaval → Castillo San Juan Bautista (16 min)
9. A pie Castillo San Juan Bautista → Auditorio de Tenerife Adán Martín (3 min)
10. Tranvía LL1 - Linea 1: Auditorio de Tenerife Adán Martín → Calle Castillo (11 min)
11. A pie Calle Castillo → Plaza del Chicharro (3 min)
12. A pie Plaza del Chicharro → Plaza de España (

In [ ]:
r = resultados[150]
draw(resultados[150], titulo="Ruta (150 ejecuciones)")


=== Resumen ruta Ruta (150 ejecuciones) ===
1. A pie Hotel (Hotel Atlántico Centro (Santa Cruz)) → Plaza del Chicharro (2 min)
2. A pie Plaza del Chicharro → Plaza Príncipe de Asturias (2 min)
3. A pie Plaza Príncipe de Asturias → Parroquia de San Francisco de Asís (2 min)
4. A pie Parroquia de San Francisco de Asís → Castillo de San Cristóbal (5 min)
5. A pie Castillo de San Cristóbal → Plaza de España (0 min)
6. Guagua L910 - INTERCAMBIADOR  SAN ANDRÉS PLAYA DE LAS TERESITAS: Plaza de España → Castillo San Juan Bautista (18 min)
7. Guagua L105 - SANTA CRUZ PUNTA DEL HIDALGO POR LA LAGUNA: Castillo San Juan Bautista → Iglesia Santo Domingo de Guzmán (31 min)
8. A pie Iglesia Santo Domingo de Guzmán → Ermita de San Miguel (3 min)
9. A pie Ermita de San Miguel → Plaza del Cristo de La Laguna (10 min)
10. A pie Plaza del Cristo de La Laguna → Casa Salazar (10 min)
11. A pie Casa Salazar → Calle San Agustín (2 min)
12. A pie Calle San Agustín → Parroquia Matriz de Nuestra Señora de la Co

Añadimos la posibilidad de planificar varios días

In [ ]:
reco = RouteRecommender(VNSStrategy())
estancia = reco.generate_stay(
    "16072026",
    horas=[5, 8, 8, 7],          # día 1: 5h, día 2: 8h, día 3: 8h, día 4: 7h
    municipio="Ambos",
    hotel="Hotel Barceló Contemporáneo (Santa Cruz)"

)

Hotel seleccionado: Hotel Barceló Contemporáneo (Santa Cruz)  (28.4739846389, -16.2528175094)
POI más cercano del municipio destino: 0.34 km
Hotel CERCA: se empieza caminando (sin guagua interurbana)
Origen = hotel. Parada más cercana: 9162
Origen: Hotel (Hotel Barceló Contemporáneo (Santa Cruz))  |  tiempo_ida = 0.0 min
DÍA 1 de 4  -  16/07/2026  (5 h)
POIs disponibles para hoy: 37
Construyendo matriz: 11 franjas × 38 POIs
  franja 0/11
  franja 5/11
  franja 10/11
Score: 40.6  (VNS)
-------------------------------------------------------
Ruta: 0, 20, 5, 2, 21, 16, 14, 11, 18, 19, 0

Puntos que se visitan:
  0  Origen
  20  Plaza Weyler
  5  Iglesia Matriz de Nuestra Señora de la Concepción
  2  Mercado Ntra. Señora de África
  21  Calle Castillo
  16  Plaza del Chicharro
  14  Plaza de la Candelaria
  11  Plaza de España
  18  Parroquia de San Francisco de Asís
  19  Plaza Príncipe de Asturias

Duración en Santa Cruz: 296 min (4.9 h)
Con ida + vuelta al hotel: 296 min (4.9 h)
DÍA 2 d

In [ ]:
draw(estancia[0], titulo="Día 1")


=== Resumen ruta Día 1 ===
1. A pie Hotel (Hotel Barceló Contemporáneo (Santa Cruz)) → Plaza Weyler (15 min)
2. Tranvía LL1 - Linea 1: Plaza Weyler → Iglesia Matriz de Nuestra Señora de la Concepción (6 min)
3. A pie Iglesia Matriz de Nuestra Señora de la Concepción → Mercado Ntra. Señora de África (7 min)
4. A pie Mercado Ntra. Señora de África → Calle Castillo (6 min)
5. A pie Calle Castillo → Plaza del Chicharro (3 min)
6. A pie Plaza del Chicharro → Plaza de la Candelaria (6 min)
7. A pie Plaza de la Candelaria → Plaza de España (1 min)
8. A pie Plaza de España → Parroquia de San Francisco de Asís (5 min)
9. A pie Parroquia de San Francisco de Asís → Plaza Príncipe de Asturias (2 min)
10. A pie Plaza Príncipe de Asturias → Hotel (Hotel Barceló Contemporáneo (Santa Cruz)) (14 min)


In [ ]:
draw(estancia[1], titulo="Día 2")


=== Resumen ruta Día 2 ===
1. A pie Hotel (Hotel Barceló Contemporáneo (Santa Cruz)) → Mirador los Campitos (15 min)
2. A pie Mirador los Campitos → Museo Militar de Almeida (22 min)
3. Guagua L946 - INTERCAMBIADOR  SAN ANDRES  TAGANANA  ALMÁCIGA: Museo Militar de Almeida → Muellito de San Andrés (18 min)
4. Guagua L910 - INTERCAMBIADOR  SAN ANDRÉS PLAYA DE LAS TERESITAS: Muellito de San Andrés → Castillo de San Cristóbal (27 min)
5. Guagua L910 - INTERCAMBIADOR  SAN ANDRÉS PLAYA DE LAS TERESITAS: Castillo de San Cristóbal → Castillo San Juan Bautista (16 min)
6. A pie Castillo San Juan Bautista → Auditorio de Tenerife Adán Martín (3 min)
7. Guagua L26 - SANTA CRUZ - LA LAGUNA  POR BARRIO LA SALUD Y FINCA ESPAÑA: Auditorio de Tenerife Adán Martín → Casa del Carnaval (21 min)
8. A pie Casa del Carnaval → Parque García Sanabria (16 min)
9. A pie Parque García Sanabria → Hotel (Hotel Barceló Contemporáneo (Santa Cruz)) (5 min)


In [ ]:
if len(estancia) >= 2:
    draw(estancia[2], titulo="Día 3")
else:
    print(f"La estancia solo generó {len(estancia)} día(s) con ruta.")


=== Resumen ruta Día 3 ===
1. Guagua L921 - INTERCAMBIADOR AVENIDA REYES CATÓLICOS INTERCAMBIADOR: Hotel (Hotel Barceló Contemporáneo (Santa Cruz)) → Museo de la Naturaleza y la Arqueología (16 min)
2. A pie Museo de la Naturaleza y la Arqueología → TEA Tenerife Espacio de las Artes (3 min)
3. A pie TEA Tenerife Espacio de las Artes → Palmetum (28 min)
4. A pie Palmetum → Parque Marítimo César Manrique (7 min)
5. Guagua L921 - INTERCAMBIADOR AVENIDA REYES CATÓLICOS INTERCAMBIADOR: Parque Marítimo César Manrique → Hotel (Hotel Barceló Contemporáneo (Santa Cruz)) (26 min)


In [ ]:
if len(estancia) > 3:
    draw(estancia[3], titulo="Día 4")
else:
    print(f"La estancia solo generó {len(estancia)} día(s) con ruta.")

La estancia solo generó 3 día(s) con ruta.


Ruta al borrar el poi

In [ ]:
res_modificado = reco.replan_without(
    resultados[20],
    ["Auditorio de Tenerife Adán Martín"]
)

print_route(
    res_modificado["ruta"],
    res_modificado["time_matrix"],
    res_modificado["init_urbano"],
    res_modificado["max_urbano"],
    tiempo_ida=res_modificado["tiempo_ida"]
)

Ruta: 0, 19, 1, 17, 20, 5, 2, 8, 15, 21, 16, 11, 7, 14, 18, 0

Puntos que se visitan:
  0  Origen
  19  Plaza Príncipe de Asturias
  1  Parque García Sanabria
  17  Mirador los Campitos
  20  Plaza Weyler
  5  Iglesia Matriz de Nuestra Señora de la Concepción
  2  Mercado Ntra. Señora de África
  8  Casa del Carnaval
  15  Castillo San Juan Bautista
  21  Calle Castillo
  16  Plaza del Chicharro
  11  Plaza de España
  7  Castillo de San Cristóbal
  14  Plaza de la Candelaria
  18  Parroquia de San Francisco de Asís

Duración en Santa Cruz: 545 min (9.1 h)
Con ida + vuelta al hotel: 545 min (9.1 h)


In [ ]:
draw(res_modificado, titulo="Ruta sin el Auditorio")


=== Resumen ruta Ruta sin el Auditorio ===
1. A pie Hotel (Hotel Barceló Contemporáneo (Santa Cruz)) → Plaza Príncipe de Asturias (3 min)
2. A pie Plaza Príncipe de Asturias → Parque García Sanabria (12 min)
3. A pie Parque García Sanabria → Mirador los Campitos (17 min)
4. A pie Mirador los Campitos → Plaza Weyler (21 min)
5. Tranvía LL1 - Linea 1: Plaza Weyler → Iglesia Matriz de Nuestra Señora de la Concepción (6 min)
6. A pie Iglesia Matriz de Nuestra Señora de la Concepción → Mercado Ntra. Señora de África (7 min)
7. A pie Mercado Ntra. Señora de África → Casa del Carnaval (9 min)
8. Guagua L919 - INTERCAMBIADOR BARRIO SALUD CUESTA PIEDRA  POR BARRANCO DE SANTOS: Casa del Carnaval → Castillo San Juan Bautista (16 min)
9. Guagua L902 - INTERCAMBIADOR  PLAZA LOS PATOS  BARRIO NUEVO: Castillo San Juan Bautista → Calle Castillo (17 min)
10. A pie Calle Castillo → Plaza del Chicharro (3 min)
11. A pie Plaza del Chicharro → Plaza de España (7 min)
12. A pie Plaza de España → Castillo d

Ruta si quiero añadir un Poi que tiene que el turista quiere visitar

In [ ]:
res_modificado2 = reco.replan_with(resultados[150], ["Auditorio de Tenerife"])
print_route(
    res_modificado2["ruta"],
    res_modificado2["time_matrix"],
    res_modificado2["init_urbano"],
    res_modificado2["max_urbano"],
    res_modificado2["tiempo_ida"]
  )

# Se puede hacer de dos formas, pasándole una ruta ya calculada o con la posibilidad de excluirlo/incluirlo antes de calcularla
# reco = RouteRecommender()
# res = reco.generate("16072026", horas=7, municipio="Santa Cruz",
                  # hotel="Hotel Barceló Contemporáneo (Santa Cruz)",
                  # excluir=["Auditorio de Tenerife Adán Martín"])


Ruta: 0, 16, 19, 18, 7, 11, 15, 10, 31, 28, 37, 30, 35, 24, 33, 32, 34, 20, 21, 5, 14, 0

Puntos que se visitan:
  0  Origen
  16  Plaza del Chicharro
  19  Plaza Príncipe de Asturias
  18  Parroquia de San Francisco de Asís
  7  Castillo de San Cristóbal
  11  Plaza de España
  15  Castillo San Juan Bautista
  10  Auditorio de Tenerife Adán Martín
  31  Iglesia Santo Domingo de Guzmán
  28  Ermita de San Miguel
  37  Plaza del Cristo de La Laguna
  30  Casa Salazar
  35  Calle San Agustín
  24  Parroquia Matriz de Nuestra Señora de la Concepción La Laguna
  33  Calle Obispo Rey Redondo
  32  Casa de los Capitanes Generales
  34  Calle Herradores
  20  Plaza Weyler
  21  Calle Castillo
  5  Iglesia Matriz de Nuestra Señora de la Concepción
  14  Plaza de la Candelaria

Duración en Santa Cruz: 631 min (10.5 h)
Con ida + vuelta al hotel: 631 min (10.5 h)


In [ ]:
draw(res_modificado2, titulo="Ruta añadiendo el Auditorio")


=== Resumen ruta Ruta añadiendo el Auditorio ===
1. A pie Hotel (Hotel Barceló Contemporáneo (Santa Cruz)) → Plaza del Chicharro (2 min)
2. A pie Plaza del Chicharro → Plaza Príncipe de Asturias (2 min)
3. A pie Plaza Príncipe de Asturias → Parroquia de San Francisco de Asís (2 min)
4. A pie Parroquia de San Francisco de Asís → Castillo de San Cristóbal (5 min)
5. A pie Castillo de San Cristóbal → Plaza de España (0 min)
6. Guagua L910 - INTERCAMBIADOR  SAN ANDRÉS PLAYA DE LAS TERESITAS: Plaza de España → Castillo San Juan Bautista (18 min)
7. A pie Castillo San Juan Bautista → Auditorio de Tenerife Adán Martín (3 min)
8. Guagua L105 - SANTA CRUZ PUNTA DEL HIDALGO POR LA LAGUNA: Auditorio de Tenerife Adán Martín → Iglesia Santo Domingo de Guzmán (34 min)
9. A pie Iglesia Santo Domingo de Guzmán → Ermita de San Miguel (3 min)
10. A pie Ermita de San Miguel → Plaza del Cristo de La Laguna (10 min)
11. A pie Plaza del Cristo de La Laguna → Casa Salazar (10 min)
12. A pie Casa Salazar → C